# Phase 3 (suite) — Deep learning séquentiel : LSTM et CNN 1D (FD001 puis FD002–FD004)

Le cahier des charges (section 5, Phase 3) prévoit, après les modèles à arbres, un **LSTM** (obligatoire) et un
**CNN 1D** (fortement recommandé). Ce notebook répond à une seule question, avec des preuves chiffrées et
reproductibles :

> **Un réseau de neurones qui lit les 30 derniers cycles d'un moteur prédit-il la RUL mieux que XGBoost ?**

Comparaison honnête : **mêmes données préparées** (Phases 1-2), **mêmes 5 plis** de validation croisée que la
Phase 4, **mêmes métriques** (RMSE, MAE, score NASA, surestimations > 20 cycles, alertes manquées / fausses),
**tests statistiques appariés** par moteur, et le jeu de test officiel évalué **une seule fois**, à la fin.

L'équipe découvre le deep learning : la section B explique **chaque notion avant son code**, une à la fois, avec
un exemple tiré de nos moteurs (fenêtre glissante, tenseur 3D, neurone, perte, gradient, époque, lot, LSTM,
convolution 1D, dropout, arrêt précoce…).

Plan : A. Préparation — B. Comprendre la méthode — C. Choix des entrées — D. Validation croisée FD001 —
E. Variante « perte asymétrique » — F. Modèle final et test officiel FD001 — G. FD002 à FD004 — H. Synthèse.

**Calcul lourd hors du notebook.** Entraîner un réseau prend une à plusieurs minutes (contre quelques
secondes pour XGBoost) : environ 80 apprentissages au total. Ils sont faits par
`scripts/entrainer_deep_learning.py`, qui écrit ses résultats dans `docs/resultats/03b_*`. Ce notebook **relit**
ces résultats (`RELANCER = False`) et ne réentraîne que les modèles à arbres (quelques secondes) : il s'exécute
de bout en bout en quelques minutes.

## A. Préparation

### A.1 Règles méthodologiques (fixées avant tout résultat)

1. **Le jeu de test ne sert jamais à choisir quoi que ce soit** : ni les entrées du réseau, ni le nombre
   d'époques, ni l'architecture. Tout choix se fait sur des moteurs de validation tirés du **train**.
2. **Les mêmes 5 plis que la Phase 4** (`calculer_plis` : `GroupKFold`, mélangé, graine 42) : un réseau et un
   XGBoost sont jugés sur exactement les mêmes moteurs de validation.
3. **Une prédiction par ligne (cycle)**, dans le même ordre que les lignes du train : les prédictions hors-pli
   des réseaux s'alignent ligne à ligne sur celles des arbres, ce qui permet les tests appariés.
4. **Réglages des réseaux fixés à l'avance, non optimisés** (pas de recherche Optuna pour les réseaux, faute de
   temps de calcul). La comparaison avec XGBoost Phase 3 (réglages de base) est équitable ; celle avec
   XGBoost retenu (choisi parmi 160 essais Optuna) avantage XGBoost.
5. **Statuts honnêtes** : si le deep learning ne bat pas XGBoost, c'est écrit tel quel.

### A.2 Imports et réglages

Les fonctions de deep learning sont dans `src/deep_learning.py` (testées par `tests/test_deep_learning.py`) ;
les métriques, les plis et les tests appariés viennent de `src/evaluation.py`, comme en Phase 4.

In [ ]:
import sys
sys.path.insert(0, "..")  # pour pouvoir importer src/ depuis notebooks/

import json
import os
import subprocess
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ordre volontaire : les modules du projet (qui chargent xgboost et lightgbm) AVANT torch. Si PyTorch est charge en
# premier, XGBoost plante ensuite dans le meme processus (conflit entre bibliotheques de calcul parallele OpenMP,
# constate le 2026-11-04 ; explication en commentaire en tete de src/deep_learning.py).
from src.config_modele import CONFIG_PHASE3, charger_config, construire_fabrique_depuis_config
from src.contrat_predictions import lire_predictions
from src.deep_learning import (
    ECHELLE_CIBLE,
    LONGUEUR_FENETRE_DEFAUT,
    ajouter_time_cycles_echelle,
    compter_parametres,
    configurer_torch,
    construire_fenetres,
    creer_modele,
    entrainer_modele,
    lister_colonnes_entree,
    predire,
    separer_moteurs,
)
from src.evaluation import (
    calculer_plis,
    comparer_apparie,
    erreurs_par_moteur,
    evaluer_en_validation_croisee,
    evaluer_sur_test,
    rmse,
    score_nasa,
)
from src.features import preparer_test, preparer_train
from src.sous_jeux import charger_sous_jeu

import torch  # apres les modules du projet (voir ci-dessus)

Les réglages du notebook : relire ou relancer le calcul lourd, seuils de lecture (les mêmes qu'en Phase 4),
dossiers de sortie, nombre de cœurs de PyTorch et couleurs des figures.

In [ ]:
# False : on relit les resultats de scripts/entrainer_deep_learning.py (docs/resultats/03b_*), quelques minutes.
# True  : on relance d'abord le script complet (23 minutes mesurees, puis 20 minutes pour la validation
#         croisee complementaire de FD002-FD004, avec 6 processus sur la machine de developpement).
RELANCER = False

LONGUEUR_FENETRE = LONGUEUR_FENETRE_DEFAUT  # 30 cycles (justifie en section B.5)
SEUIL_SURESTIMATION = 20  # cycles : surestimation dangereuse (choix de la Phase 3)
SEUIL_ALERTE = 30         # cycles : seuil d'envoi en maintenance, HYPOTHESE DE TRAVAIL
PLAFOND_RUL = 125         # plafond de la RUL (Phase 2)

DOSSIER_RESULTATS = os.path.join("..", "docs", "resultats")
DOSSIER_FIGURES = os.path.join("..", "docs", "figures")
os.makedirs(DOSSIER_FIGURES, exist_ok=True)

# PyTorch sur 1 coeur (reglage de src/deep_learning.py, rappele ici) : ce notebook fait aussi calculer XGBoost dans
# le meme processus, et un calcul PyTorch multi-coeurs peut alors se bloquer (conflit entre deux bibliotheques
# OpenMP, constate le 2026-11-04 dans la suite pytest). 1 coeur suffit pour les petites demonstrations ci-dessous.
configurer_torch(1)

# memes couleurs que le notebook 04 (teintes distinctes, lisibles par les daltoniens)
COULEUR_1 = "#2a78d6"  # bleu : XGBoost Phase 3
COULEUR_4 = "#8fb8ea"  # bleu clair : XGBoost retenu
COULEUR_2 = "#eb6834"  # orange : LSTM
COULEUR_3 = "#1baf7a"  # vert d'eau : CNN 1D
COULEUR_TEXTE = "#52514e"  # gris fonce (reperes, annotations)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.axisbelow"] = True
plt.rcParams["grid.color"] = "#e4e3df"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
debut_notebook = time.perf_counter()  # chronometre de l'execution complete du notebook

### A.3 Relancer le calcul lourd (facultatif)

Avec `RELANCER = True`, la cellule suivante relance `scripts/entrainer_deep_learning.py` sur les quatre
sous-jeux (puis la validation croisée complémentaire de FD002–FD004, section G) : il réécrit tous les fichiers `docs/resultats/03b_*`, les fichiers de prédictions
(`sorties/predictions/FD00X__lstm.*`, `FD00X__cnn1d.*`) et les poids (`models/FD00X__lstm.pt`, `__cnn1d.pt`).
À graine égale, les chiffres obtenus sont identiques au bit près sur la même machine (seules les durées changent).

In [ ]:
if RELANCER:
    chemin_script = os.path.join("..", "scripts", "entrainer_deep_learning.py")
    subprocess.run([sys.executable, chemin_script, "--sous-jeux", "FD001", "FD002", "FD003", "FD004"], check=True)
    subprocess.run([sys.executable, chemin_script, "--sous-jeux", "FD002", "FD003", "FD004", "--seulement-cv-sous-jeux"],
                   check=True)
    subprocess.run([sys.executable, chemin_script, "--seulement-resume"], check=True)
else:
    print("RELANCER = False : on relit les resultats deja calcules dans docs/resultats/03b_*.")

### A.4 Données FD001 et plis

Exactement la préparation des Phases 1 à 3 (`preparer_train` / `preparer_test`) : capteurs constants retirés,
min-max appris sur le train, lissage sur 5 cycles moteur par moteur, RUL plafonnée à 125. Les réseaux utilisent
les **15 capteurs normalisés puis lissés** (pas les moyennes et écarts-types glissants des arbres : le réseau voit
lui-même les 30 derniers cycles). On ajoute la colonne `time_cycles_echelle` = `time_cycles` / 100 (section C).
Les plis sont calculés une seule fois, comme en Phase 4.

In [ ]:
contexte_train = preparer_train(fenetre=5, plafond=PLAFOND_RUL)  # pipeline des Phases 1 a 3 (src/features.py)
contexte_test = preparer_test(contexte_train)  # memes etapes sur le test, sans rien reapprendre

train = ajouter_time_cycles_echelle(contexte_train["train"])  # ajoute time_cycles / 100
test = ajouter_time_cycles_echelle(contexte_test["test"])
capteurs = contexte_train["sensor_cols_restants"]  # les 15 capteurs normalises puis lisses

X = contexte_train["X"]            # les 46 features des arbres (pour les XGBoost de comparaison)
y = contexte_train["y"]            # cible : RUL plafonnee a 125
groups = contexte_train["groups"]  # numero de moteur de chaque ligne
X_test_dernier = contexte_test["X_test_dernier"]  # features du dernier cycle de chaque moteur test
y_test_vraie = contexte_test["y_test_vraie"]      # RUL vraie brute (RUL_FD001.txt)
moteurs_test = contexte_test["moteurs_test"]

plis = calculer_plis(X, y, groups, n_splits=5, random_state=42)  # les plis de la Phase 4
NB_MOTEURS_TRAIN = int(groups.nunique())

print(f"Train : {len(train)} lignes, {NB_MOTEURS_TRAIN} moteurs ; test : {len(test)} lignes, {len(moteurs_test)} moteurs")
print(f"Capteurs utilises par les reseaux ({len(capteurs)}) : {capteurs}")

### A.5 Les références XGBoost, calculées en premier

Les réseaux seront comparés à **XGBoost Phase 3** (réglages de base) et à **XGBoost retenu** (Phase 4,
`docs/resultats/04_config_retenue.json`). On les réentraîne ici (quelques secondes) pour disposer de leurs
prédictions hors-pli (validation croisée, mêmes plis) et de leurs prédictions sur le test.

**Pourquoi maintenant, avant tout réseau ?** Une contrainte technique, constatée en préparant ce notebook :
xgboost / lightgbm, scikit-learn et PyTorch embarquent chacun leur propre copie de la bibliothèque de calcul
parallèle OpenMP, et ces copies cohabitent mal dans un même processus Python :

- si PyTorch est **chargé** avant xgboost, XGBoost plante ensuite dès qu'il calcule sur plusieurs cœurs (erreur de
  segmentation, reproduite 3 fois sur 3) ; chargé après, il fonctionne. D'où l'ordre des imports de la cellule A.2 ;
- si XGBoost a déjà calculé, PyTorch sur **plusieurs** cœurs peut se bloquer indéfiniment (constaté dans la suite
  de tests) ; sur 1 cœur, non. D'où le réglage à 1 cœur.

Par prudence, on fait aussi tous les calculs XGBoost avant d'utiliser PyTorch. Les chiffres n'en dépendent pas :
seul l'ordre des opérations change (contrôles de reproduction en D.2 et F.2).

In [ ]:
config_retenue = charger_config()  # docs/resultats/04_config_retenue.json
configs_xgb = {}
configs_xgb["XGBoost Phase 3"] = CONFIG_PHASE3
configs_xgb["XGBoost retenu"] = config_retenue

resultats_xgb = {}  # validation croisee (memes plis)
resultats_test_xgb = {}  # test officiel, modele entraine sur les 100 moteurs
temps_prediction_xgb_ms = {}  # prediction de toutes les trajectoires test, ramenee a un moteur
X_test_tous_cycles = contexte_test["test"][contexte_train["feature_cols"]]  # toutes les lignes du test
for nom_modele in configs_xgb:
    fabrique = construire_fabrique_depuis_config(configs_xgb[nom_modele])
    resultats_xgb[nom_modele] = evaluer_en_validation_croisee(fabrique, X, y, groups, plis,
                                                              seuil_surestimation=SEUIL_SURESTIMATION,
                                                              seuil_alerte=SEUIL_ALERTE)
    modele_final = fabrique()
    modele_final.fit(X, y)  # 100 moteurs d'entrainement
    resultats_test_xgb[nom_modele] = evaluer_sur_test(modele_final, X_test_dernier, y_test_vraie, plafond=PLAFOND_RUL,
                                                      seuil_surestimation=SEUIL_SURESTIMATION, seuil_alerte=SEUIL_ALERTE)
    debut = time.perf_counter()
    modele_final.predict(X_test_tous_cycles)
    temps_prediction_xgb_ms[nom_modele] = 1000.0 * (time.perf_counter() - debut) / len(moteurs_test)
    print(f"{nom_modele} : RMSE CV moyen = {resultats_xgb[nom_modele]['rmse_moyen']:.3f}, "
          f"RMSE test = {resultats_test_xgb[nom_modele]['rmse']:.2f}")

## B. Comprendre la méthode avant le code

Ce qui change par rapport aux arbres de la Phase 3 : XGBoost reçoit **une ligne** par cycle (les capteurs du
cycle, des statistiques sur 5 cycles, `time_cycles`) et ne sait rien de l'ordre des cycles. Un réseau
séquentiel reçoit **une petite série temporelle** : les 30 derniers cycles du moteur, dans l'ordre, et il apprend
lui-même quels motifs annoncent la panne (par exemple « `sensor_11` monte de plus en plus vite »).

On introduit les notions une par une. Chaque notion est suivie d'une petite cellule qui la montre sur nos données.

### B.1 Notion 1 — la fenêtre glissante

**Définition.** Pour prédire la RUL au cycle *t* d'un moteur, on donne au réseau la **fenêtre** des 30 derniers
cycles **de ce moteur** : cycles *t* − 29 à *t*. Au cycle suivant, la fenêtre « glisse » d'un cycle.

**Exemple concret.** Moteur 1 du train (192 cycles), au cycle 50 : la fenêtre contient les cycles 21 à 50. La
cible associée est la RUL de ce cycle : 192 − 50 = 142, plafonnée à 125.

Deux garanties, vérifiées plus bas et par `tests/test_deep_learning.py` : **jamais de cycle futur** (la fenêtre
s'arrête au cycle *t*) et **jamais un autre moteur**. Et **une fenêtre par ligne**, dans l'ordre des lignes : la
prédiction n° *i* correspond à la ligne n° *i* du tableau, exactement comme pour XGBoost.

Pour « voir » ce que contient une fenêtre, on construit les fenêtres du moteur 1 avec une seule entrée un peu
spéciale : le numéro de cycle lui-même.

In [ ]:
moteur_1 = train[train["unit_number"] == 1]  # les 192 cycles du moteur 1
# entree = time_cycles lui-meme : chaque fenetre affiche donc directement les numeros de cycles qu'elle contient
fenetres_cycles, _, _, cycles_presents = construire_fenetres(moteur_1, ["time_cycles"], longueur=LONGUEUR_FENETRE)

position_cycle_50 = int(np.flatnonzero(cycles_presents == 50)[0])  # la ligne du cycle 50
fenetre_50 = fenetres_cycles[position_cycle_50, :, 0]
print(f"Forme des fenetres du moteur 1 : {fenetres_cycles.shape} (une fenetre par cycle, 30 pas, 1 entree)")
print(f"Fenetre du cycle 50 : cycles {int(fenetre_50[0])} a {int(fenetre_50[-1])}")
print(fenetre_50.astype(int))
assert int(fenetre_50[0]) == 21 and int(fenetre_50[-1]) == 50

**Figure** : ce que « voit » le réseau. Le capteur `sensor_11` (normalisé et lissé) du moteur 1, avec deux
fenêtres : au cycle 50 (début de vie, RUL plafonnée à 125) et au cycle 180 (fin de vie, RUL de 12 cycles). Le
réseau reçoit les 15 capteurs de la zone grisée, et doit sortir le nombre écrit à droite.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(moteur_1["time_cycles"], moteur_1["sensor_11"], color=COULEUR_1, linewidth=1.8, label="sensor_11 (normalisé, lissé)")

# deux fenetres d'exemple : (cycle present, couleur, position du texte par rapport au point)
for cycle_present, couleur, decalage_texte in [(50, COULEUR_3, (6, 0.15)), (180, COULEUR_2, (-95, 0.0))]:
    rul_vraie = int(moteur_1["time_cycles"].max() - cycle_present)  # RUL brute a ce cycle
    rul_cible = min(rul_vraie, PLAFOND_RUL)  # cible plafonnee (Phase 2)
    ax.axvspan(cycle_present - LONGUEUR_FENETRE + 1, cycle_present, color=couleur, alpha=0.18)
    valeur = float(moteur_1.loc[moteur_1["time_cycles"] == cycle_present, "sensor_11"].iloc[0])
    position_texte = (cycle_present + decalage_texte[0], valeur + decalage_texte[1])
    ax.annotate(f"fenêtre du cycle {cycle_present}\n(cycles {cycle_present - LONGUEUR_FENETRE + 1} à {cycle_present})\n"
                f"cible : RUL = {rul_cible}", xy=(cycle_present, valeur), xytext=position_texte,
                fontsize=9, color=COULEUR_TEXTE, arrowprops={"arrowstyle": "->", "color": COULEUR_TEXTE})

ax.set_xlabel("cycle du moteur 1 (train FD001)")
ax.set_ylabel("sensor_11 (0 = min, 1 = max du train)")
ax.set_title("Fenêtre glissante : le réseau voit les 30 derniers cycles et prédit la RUL du dernier")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(DOSSIER_FIGURES, "03b_fenetre_glissante_moteur1.png"), dpi=150)
plt.show()

### B.2 Notion 2 — le remplissage en début de vie

**Problème.** Au cycle 3, le moteur n'a que 3 cycles observés : impossible de remplir une fenêtre de 30.

**Choix.** On complète le **début** de la fenêtre en **répétant le premier cycle** du moteur : la fenêtre du
cycle 3 est [cycle 1, cycle 1, …, cycle 1, cycle 2, cycle 3]. Pourquoi pas des zéros ? Pour un capteur
normalisé, 0 est une vraie valeur (le minimum du train) : le réseau croirait voir un moteur dans un état
extrême. Répéter le premier cycle dit seulement « rien n'a changé avant », ce qui est le plus neutre pour un
moteur qui commence sa vie. On n'emprunte **jamais** de cycles à un autre moteur.

Ce remplissage ne touche que les 29 premiers cycles de chaque moteur. Sur le test FD001, le dernier cycle (celui
qui est noté) a toujours au moins 31 cycles derrière lui : pas de remplissage à cet endroit (section B.5).

In [ ]:
position_cycle_3 = int(np.flatnonzero(cycles_presents == 3)[0])
fenetre_3 = fenetres_cycles[position_cycle_3, :, 0]
print("Fenetre du cycle 3 :", fenetre_3.astype(int))

nb_cycle_1 = int(np.sum(fenetre_3 == 1))  # positions qui contiennent le cycle 1
print(f"{nb_cycle_1} positions contiennent le cycle 1 : {nb_cycle_1 - 1} de remplissage + le vrai cycle 1 ; "
      f"puis les cycles 2 et 3.")
assert list(fenetre_3[-3:].astype(int)) == [1, 2, 3]

### B.3 Contrôle sur tout le train : jamais un autre moteur, jamais le futur

Même astuce sur les 20 631 lignes du train, avec deux entrées « étiquettes » : `unit_number` et `time_cycles`.
Pour chaque fenêtre, on vérifie que **toutes** ses positions viennent du moteur de la ligne, et d'un cycle
inférieur ou égal au cycle de la ligne.

In [ ]:
fenetres_controle, _, numeros_lignes, cycles_lignes = construire_fenetres(train, ["unit_number", "time_cycles"],
                                                                         longueur=LONGUEUR_FENETRE)
moteurs_vus = fenetres_controle[:, :, 0]  # (20631, 30) : le moteur de chaque position de chaque fenetre
cycles_vus = fenetres_controle[:, :, 1]   # (20631, 30) : le cycle de chaque position

meme_moteur = moteurs_vus == numeros_lignes[:, np.newaxis]  # Vrai si la position vient du moteur de la ligne
pas_de_futur = cycles_vus <= cycles_lignes[:, np.newaxis]   # Vrai si la position est un cycle <= cycle present
dernier_pas_present = cycles_vus[:, -1] == cycles_lignes    # la derniere position est le cycle present

print(f"Positions venant d'un autre moteur : {int(np.sum(~meme_moteur))} sur {meme_moteur.size}")
print(f"Positions venant du futur          : {int(np.sum(~pas_de_futur))} sur {pas_de_futur.size}")
print(f"Fenetres dont le dernier pas n'est pas le cycle present : {int(np.sum(~dernier_pas_present))}")
assert meme_moteur.all() and pas_de_futur.all() and dernier_pas_present.all()

### B.4 Notion 3 — le tenseur 3D

**Définition.** Un tableau à 3 dimensions. Le tableau des arbres a 2 dimensions (lignes × features). Celui
des réseaux en a 3 : **(nombre de fenêtres, 30 cycles, nombre d'entrées)**. Image : une pile de 20 631 petites
feuilles de calcul, chacune de 30 lignes (les cycles) et 15 colonnes (les capteurs).

On vérifie que la dernière « ligne » de chaque feuille est bien la ligne correspondante du train, et que les
moteurs des fenêtres sont, dans l'ordre, ceux des lignes (alignement avec les arbres).

In [ ]:
fenetres_capteurs, y_fenetres, groupes_fenetres, _ = construire_fenetres(train, capteurs, longueur=LONGUEUR_FENETRE,
                                                                        colonne_cible="RUL_cappee")
print(f"Tenseur 3D : {fenetres_capteurs.shape} = (fenetres, cycles, capteurs), type {fenetres_capteurs.dtype}, "
      f"{fenetres_capteurs.nbytes / 1e6:.1f} Mo")

valeurs_lignes = train[capteurs].to_numpy(dtype=np.float32)
assert np.array_equal(fenetres_capteurs[:, -1, :], valeurs_lignes)  # dernier pas = la ligne elle-meme
assert np.array_equal(groupes_fenetres, groups.to_numpy())          # meme moteur, dans le meme ordre
assert np.array_equal(y_fenetres, y.to_numpy(dtype=np.float32))     # meme cible
print("Alignement ligne a ligne avec le tableau des arbres : verifie.")

### B.5 Pourquoi des fenêtres de 30 cycles ?

Un choix de données, pas de littérature : on veut que la fenêtre du **dernier cycle observé** de chaque moteur
test (le seul qui est noté) soit remplie de vraies mesures. Plus long, des moteurs test courts seraient
complétés par remplissage au moment même où on les note ; plus court, le réseau verrait moins de tendance.

In [ ]:
lignes = []
for sous_jeu in ["FD001", "FD002", "FD003", "FD004"]:  # longueur des trajectoires test de chaque sous-jeu
    brut_test = charger_sous_jeu(sous_jeu, "test")
    longueurs = brut_test.groupby("unit_number").size()  # nombre de cycles observes par moteur test
    ligne = {}
    ligne["sous_jeu"] = sous_jeu
    ligne["nb_moteurs_test"] = int(len(longueurs))
    ligne["cycles_observes_min"] = int(longueurs.min())
    ligne["cycles_observes_mediane"] = float(longueurs.median())
    ligne["moteurs_test_de_moins_de_30_cycles"] = int(np.sum(longueurs < LONGUEUR_FENETRE))
    lignes.append(ligne)
pd.DataFrame(lignes).set_index("sous_jeu")

Lecture : sur FD001, le plus court moteur test a 31 cycles — une fenêtre de 30 est toujours pleine au cycle noté.
Sur FD002 et FD004, quelques moteurs test sont plus courts (le minimum est de 21 et 19 cycles) : leur fenêtre
est complétée par remplissage (B.2). On garde 30 partout pour ne pas multiplier les réglages ; c'est une
limite connue pour ces quelques moteurs.

### B.6 Notion 4 — un neurone

**Définition.** Un neurone fait une **somme pondérée** de ses entrées, ajoute un **biais**, puis applique une
fonction simple, par exemple **ReLU** : max(0, x). Les **poids** et le biais sont les nombres que
l'apprentissage ajuste. Un **réseau** empile des **couches** de neurones : les sorties d'une couche sont les
entrées de la suivante.

**Exemple concret** avec 3 vraies valeurs du moteur 1 au cycle 150 et des poids **choisis à la main** pour
l'exemple (le réseau, lui, les apprend) :

In [ ]:
ligne_exemple = moteur_1[moteur_1["time_cycles"] == 150].iloc[0]  # moteur 1, cycle 150 (RUL vraie = 42)
noms_entrees = ["sensor_11", "sensor_4", "time_cycles_echelle"]
entrees = ligne_exemple[noms_entrees].to_numpy(dtype=float)
poids = np.array([-60.0, -50.0, -20.0])  # choisis a la main pour l'exemple
biais = 140.0

somme = 0.0
for position in range(len(entrees)):  # somme ponderee, terme par terme
    terme = poids[position] * entrees[position]
    print(f"  {noms_entrees[position]:20s} = {entrees[position]:.3f}  x poids {poids[position]:+.0f}  = {terme:+.2f}")
    somme = somme + terme
somme = somme + biais
sortie = max(0.0, somme)  # ReLU : les valeurs negatives deviennent 0
print(f"somme + biais ({biais:.0f}) = {somme:.2f} -> ReLU -> sortie du neurone = {sortie:.2f}")

### B.7 Notion 5 — la perte et le gradient : apprendre, c'est descendre une pente

**La perte** mesure à quel point les prédictions sont fausses. On utilise l'**erreur quadratique moyenne**
(MSE) : la moyenne de (prédiction − vérité)². C'est la perte native de XGBoost Phase 3.

**Le gradient** dit, pour chaque poids, dans quel sens il faut le bouger pour que la perte baisse, et à quel
point. L'apprentissage répète : prédire → calculer la perte → calculer le gradient → bouger chaque poids d'un
petit pas dans le bon sens.

**Exemple minimal** : un « réseau » à un seul poids *w* qui prédit toujours *w*, pour un moteur dont la vraie
RUL est 100. Perte = (*w* − 100)² ; gradient = 2 (*w* − 100). On part de *w* = 80 avec un pas de 0,1 :

In [ ]:
w = 80.0  # le poids de depart
pas = 0.1  # le "taux d'apprentissage"
vraie_rul = 100.0
for etape in range(1, 6):  # 5 pas de descente de gradient
    perte = (w - vraie_rul) ** 2
    gradient = 2.0 * (w - vraie_rul)  # negatif : il faut augmenter w
    w = w - pas * gradient  # on bouge w dans le sens qui fait baisser la perte
    print(f"etape {etape} : perte avant = {perte:6.1f}, gradient = {gradient:+6.1f}, nouveau w = {w:.2f}")

Le poids se rapproche de 100 et la perte baisse à chaque pas. Un vrai réseau fait la même chose avec ses
54 000 poids à la fois ; PyTorch calcule tous les gradients automatiquement (`perte.backward()`).

**Adam** est la méthode utilisée pour faire ces pas : c'est une descente de gradient qui adapte la taille du pas
de chaque poids à l'historique de ses gradients. Réglage : le pas de base par défaut de PyTorch, 0,001.

### B.8 Notion 6 — la normalisation de la cible

Les capteurs sont entre 0 et 1 (min-max, Phase 1), mais la RUL va de 0 à 125. Un réseau apprend mieux quand
entrées et sortie ont des ordres de grandeur proches : des sorties de l'ordre de 100 demanderaient des poids
très grands, et les premiers gradients seraient énormes. On **divise donc la cible par 125** pendant
l'apprentissage (elle va de 0 à 1), et on **remultiplie par 125** les prédictions. Une prédiction négative est
ramenée à 0 (une RUL négative n'a pas de sens physique). Tout cela est caché dans `src/deep_learning.py` : on
donne et on récupère des RUL en cycles.

In [ ]:
cible = y.to_numpy(dtype=float)
cible_echelle = cible / ECHELLE_CIBLE  # ce que le reseau apprend a sortir
print(f"RUL plafonnee : de {cible.min():.0f} a {cible.max():.0f} cycles -> cible du reseau : de {cible_echelle.min():.2f} "
      f"a {cible_echelle.max():.2f}")
print(f"Exemple : moteur 1, cycle 150, RUL 42 -> le reseau doit sortir 42 / 125 = {42 / ECHELLE_CIBLE:.3f}")

### B.9 Notion 7 — l'époque et le lot

On ne calcule pas le gradient sur les 20 000 fenêtres à la fois (trop lent) ni sur une seule (trop bruité) :
on les découpe en **lots** de 256 fenêtres tirées au hasard. Chaque lot donne **une** mise à jour des poids.
Une **époque** = un passage complet sur toutes les fenêtres d'apprentissage, dans un nouvel ordre aléatoire.

Exemple avec le pli 1 de la validation croisée (section D) : combien de lots par époque ?

In [ ]:
idx_train_pli_1 = plis[0][0]  # lignes d'entrainement du pli 1
moteurs_pli_1 = np.unique(groups.to_numpy()[idx_train_pli_1])
moteurs_apprentissage_1, moteurs_arret_1 = separer_moteurs(moteurs_pli_1, fraction_validation=0.2, graine=42)
nb_fenetres_apprentissage = int(np.sum(np.isin(groups.to_numpy(), moteurs_apprentissage_1)))
nb_lots = int(np.ceil(nb_fenetres_apprentissage / 256))
print(f"Pli 1 : {len(moteurs_pli_1)} moteurs d'entrainement = {len(moteurs_apprentissage_1)} d'apprentissage "
      f"+ {len(moteurs_arret_1)} d'arret precoce (section B.13)")
print(f"{nb_fenetres_apprentissage} fenetres d'apprentissage / 256 = {nb_lots} lots, donc {nb_lots} mises a jour "
      f"des poids par epoque")

### B.10 Notion 8 — le LSTM

**Idée.** Un LSTM (*Long Short-Term Memory*) lit la fenêtre **cycle par cycle**, du plus ancien au plus récent,
comme on lirait une phrase mot par mot. Il garde une **mémoire** de 64 nombres, mise à jour à chaque cycle à
partir du cycle lu et de la mémoire précédente. Des **portes** (petites couches de neurones apprises) décident à
chaque pas ce qu'il faut oublier, ce qu'il faut ajouter, et ce qu'il faut transmettre. Exemple de ce qu'il peut
apprendre : « si `sensor_11` augmente depuis 20 cycles, retenir que la dégradation a commencé ».

**Notre architecture** (simple et standard, fixée à l'avance) :
1. deux couches LSTM de 64 unités empilées (la 2e lit la suite des mémoires de la 1re) ;
2. on garde la sortie du **dernier** cycle (le cycle présent, qui a « vu » toute la fenêtre) ;
3. une couche linéaire 64 → 1 donne la RUL (divisée par 125).

Environ 54 000 poids, pour 20 631 fenêtres : assez pour apprendre des motifs, et le dropout (B.12) et l'arrêt
précoce (B.13) limitent le risque d'apprendre le train par cœur.

In [ ]:
modele_lstm_demo = creer_modele("lstm", n_entrees=len(capteurs), graine=42)  # reseau neuf (poids au hasard)
print(modele_lstm_demo)
print(f"Nombre de poids : {compter_parametres(modele_lstm_demo)}")

predictions_demo = predire(modele_lstm_demo, fenetres_capteurs[:4])  # 4 fenetres -> 4 RUL
print("Sorties d'un reseau NON entraine pour 4 fenetres (sans signification) :", np.round(predictions_demo, 1))

### B.11 Notion 9 — la convolution 1D

**Idée.** Un **filtre** est une petite suite de poids (par exemple 3 nombres) qu'on fait **glisser** le long des
cycles : à chaque position, on multiplie les 3 cycles par les 3 poids et on additionne. Le filtre répond fort
quand le signal ressemble au motif qu'il code. Un réseau convolutif **apprend** ses filtres.

**Exemple concret, filtre écrit à la main** sur 9 cycles : poids −4, −3, …, 0, …, +3, +4, divisés par 60. Ce
filtre calcule la **pente** de la droite qui passe au mieux par les 9 cycles (« de combien le capteur monte par
cycle »). Appliqué au `sensor_11` du moteur 1, il reste proche de 0 en début de vie et devient positif en fin de
vie, quand le capteur monte. Les filtres du CNN sont plus courts (5 et 3 cycles), portent sur les 15 capteurs à
la fois, et leurs poids sont appris au lieu d'être écrits à la main.

In [ ]:
signal = moteur_1["sensor_11"].to_numpy()  # 192 valeurs, une par cycle
filtre = (np.arange(9) - 4.0) / 60.0  # poids -4/60 ... +4/60 : pente au sens des moindres carres sur 9 cycles
largeur = len(filtre)
print("Poids du filtre :", np.round(filtre, 3))

sortie_filtre = np.zeros(len(signal) - largeur + 1)  # une valeur par position du filtre
for debut in range(len(sortie_filtre)):  # on fait glisser le filtre, une position a la fois
    somme = 0.0
    for j in range(largeur):  # produit terme a terme, puis somme
        somme = somme + filtre[j] * signal[debut + j]
    sortie_filtre[debut] = somme

print(f"Sortie moyenne du filtre sur les 50 premiers cycles : {sortie_filtre[:50].mean():+.4f}")
print(f"Sortie moyenne du filtre sur les 30 derniers cycles : {sortie_filtre[-30:].mean():+.4f}")

fig, axes = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)
axes[0].plot(moteur_1["time_cycles"], signal, color=COULEUR_1)
axes[0].set_ylabel("sensor_11")
axes[0].set_title("Moteur 1 : le signal (en haut) et la sortie du filtre « pente sur 9 cycles » (en bas)")
cycles_centres = moteur_1["time_cycles"].to_numpy()[4:-4]  # chaque sortie est placee au cycle du milieu des 9
axes[1].plot(cycles_centres, sortie_filtre, color=COULEUR_3)
axes[1].axhline(0, color=COULEUR_TEXTE, linewidth=1)
axes[1].set_ylabel("pente (par cycle)")
axes[1].set_xlabel("cycle")
plt.tight_layout()
plt.savefig(os.path.join(DOSSIER_FIGURES, "03b_convolution_exemple.png"), dpi=150)
plt.show()

### B.12 Notion 10 — du filtre au CNN 1D (BatchNorm, ReLU, pooling global)

Un **CNN 1D** applique en parallèle **beaucoup** de filtres appris, chacun sur les 15 capteurs à la fois (un filtre
de largeur 5 a 5 × 15 poids). Trois ingrédients de plus :

- **ReLU** (B.6) après chaque convolution : sans elle, empiler des convolutions reviendrait à une seule ;
- **BatchNorm** : remet la sortie de chaque filtre à une échelle stable (moyenne ≈ 0, écart-type ≈ 1 sur le
  lot), ce qui rend l'apprentissage plus rapide et moins sensible au réglage du pas ;
- **pooling global moyen** : après les convolutions, chaque filtre a une réponse par position de la fenêtre ;
  on en fait la **moyenne** sur les 30 positions → un nombre par filtre (« ce motif est-il présent dans la
  fenêtre ? »).

**Notre architecture** : 3 blocs « convolution → BatchNorm → ReLU » (32, 64 puis 64 filtres, de largeur 5, 5
et 3 cycles ; après les 3 blocs, chaque position « voit » 11 cycles), pooling global moyen, dropout, couche
linéaire 64 → 1. Environ 26 000 poids. Différence avec le LSTM : le CNN regarde des morceaux de quelques cycles
partout dans la fenêtre en même temps ; le LSTM lit la fenêtre dans l'ordre et résume tout dans sa mémoire.

In [ ]:
modele_cnn_demo = creer_modele("cnn1d", n_entrees=len(capteurs), graine=42)
print(modele_cnn_demo)
print(f"Nombre de poids : {compter_parametres(modele_cnn_demo)}")
print("Sorties pour 4 fenetres (reseau non entraine) :", np.round(predire(modele_cnn_demo, fenetres_capteurs[:4]), 1))

### B.13 Notion 11 — le dropout

**Définition.** Pendant l'apprentissage, on « éteint » au hasard une partie des neurones (ici 20 %) à chaque lot :
leur sortie est mise à 0, et les autres sont multipliées par 1 / (1 − 0,2) = 1,25 pour garder la même moyenne.
Le réseau ne peut donc pas compter sur un neurone en particulier, ce qui limite l'apprentissage par cœur. En
**prédiction**, le dropout est coupé : tous les neurones travaillent. Démonstration sur un vecteur de 10 « 1 » :

In [ ]:
couche_dropout = torch.nn.Dropout(p=0.2)
vecteur = torch.ones(10)
torch.manual_seed(0)

couche_dropout.train()  # mode apprentissage
print("En apprentissage :", couche_dropout(vecteur).numpy())
couche_dropout.eval()  # mode prediction
print("En prediction    :", couche_dropout(vecteur).numpy())

### B.14 Notion 12 — le surapprentissage et l'arrêt précoce (*early stopping*)

**Le risque.** Si on entraîne trop longtemps, le réseau finit par apprendre les **particularités des moteurs
d'entraînement** (leur bruit) plutôt que la dégradation en général : la perte d'entraînement continue de baisser,
mais l'erreur sur des moteurs **nouveaux** remonte. C'est le **surapprentissage**.

**L'arrêt précoce.** On met de côté quelques moteurs d'entraînement (20 %) qui ne servent **pas** à ajuster les
poids. Après chaque époque, on mesure l'erreur sur eux. On garde les poids de la **meilleure époque**, et on
arrête si l'erreur ne s'améliore plus pendant 10 époques (la **patience** ; 80 époques au plus).

Point crucial dans la validation croisée : ces moteurs d'arrêt précoce sont pris parmi les moteurs
d'**entraînement** du pli, **jamais** parmi ses moteurs de validation. Sinon, les moteurs qu'on prétend
« jamais vus » auraient choisi le moment de l'arrêt, et le score serait optimiste.

### B.15 Démonstration : 3 époques d'un LSTM, en direct

Pour voir le code tourner (les vrais apprentissages, jusqu'à l'arrêt précoce, sont dans le script) : 80 moteurs
d'apprentissage, 20 moteurs d'arrêt précoce, 3 époques.

In [ ]:
moteurs_app_demo, moteurs_arret_demo = separer_moteurs(np.unique(groupes_fenetres), fraction_validation=0.2, graine=42)
masque_app_demo = np.isin(groupes_fenetres, moteurs_app_demo)  # Vrai sur les fenetres des moteurs d'apprentissage

modele_demo = creer_modele("lstm", n_entrees=len(capteurs), graine=42)
historique_demo = entrainer_modele(
    modele_demo,
    fenetres_capteurs[masque_app_demo], y_fenetres[masque_app_demo],      # apprentissage
    fenetres_capteurs[~masque_app_demo], y_fenetres[~masque_app_demo],    # arret precoce
    max_epoques=3, patience=10, graine=42, verbeux=True,
)
print(f"{historique_demo['duree_s']:.1f} s pour 3 epoques ; RMSE des moteurs d'arret precoce apres chaque epoque : "
      f"{np.round(historique_demo['rmse_validation'], 2)}")

La « perte » affichée est calculée sur la cible divisée par 125 : 0,01 correspond à un RMSE d'environ
√0,01 × 125 = 12,5 cycles. Elle baisse d'une époque à l'autre : le réseau apprend.

### B.16 Notion 13 — la reproductibilité, et le choix CPU / MPS

Un réseau utilise le hasard (poids initiaux, ordre des lots, dropout). Avec des **graines** fixées, le même
apprentissage redonne exactement le même modèle. Il reste à choisir **où** calculer : le processeur (**CPU**)
ou la puce graphique du Mac (**MPS**). Mesures faites par le script (`docs/resultats/03b_cpu_vs_mps.csv`) :
vitesse en lots de 256 fenêtres par seconde, et écart maximal entre les prédictions de deux apprentissages
identiques (même graine).

In [ ]:
mesures_appareils = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "03b_cpu_vs_mps.csv"))
debits = mesures_appareils[mesures_appareils["mesure"] == "debit_lots_par_s"]
print("Vitesse (lots de 256 fenetres par seconde, apprentissage complet : avant + arriere + mise a jour) :")
print(debits.pivot_table(index="type_modele", columns=["appareil", "nb_threads_cpu"], values="valeur").round(1).to_string())
print(f"(charge moyenne de la machine pendant la mesure : {debits['charge_machine_1min'].iloc[0]:.1f} sur 15 coeurs)")

ecarts = mesures_appareils[mesures_appareils["mesure"] != "debit_lots_par_s"]
print("\nEcart maximal entre les predictions de deux apprentissages a graine egale (cycles) :")
print(ecarts.pivot_table(index="type_modele", columns="mesure", values="valeur").to_string())

**Choix : le CPU, 1 cœur par apprentissage, plusieurs apprentissages en parallèle.**

- Sur un apprentissage isolé, MPS est nettement plus rapide : environ 4 fois pour le LSTM (252 lots/s contre 59)
  et 25 fois pour le CNN 1D (389 contre 16 ; la convolution est particulièrement lente sur ce processeur).
- Mais MPS ne redonne pas les mêmes chiffres que le CPU : à graine égale, après 2 époques, jusqu'à 1 cycle d'écart
  pour le LSTM et 29 cycles pour le CNN. Le CPU, lui, redonne exactement les mêmes prédictions d'une exécution à
  l'autre, avec 1 comme avec 6 cœurs. Et MPS n'existe que sur les Mac à puce Apple : des chiffres obtenus sur MPS
  ne pourraient pas être recalculés par un coéquipier qui n'en a pas.
- 6 cœurs ne vont pas plus vite qu'un seul pour ces petits réseaux (LSTM : 41 lots/s contre 59). Le script lance donc
  jusqu'à 6 apprentissages de 1 cœur en même temps : le calcul complet FD001 à FD004 a pris 23 minutes, la vitesse
  de MPS n'était pas nécessaire.
- Raison supplémentaire, découverte ensuite (section A.5) : PyTorch sur plusieurs cœurs peut se bloquer quand
  XGBoost a déjà calculé dans le même processus ; sur 1 cœur, non.
- Coût accepté : un CNN 1D environ 25 fois plus lent qu'il pourrait l'être. Les durées des tableaux sont mesurées
  sur 1 cœur, sur une machine partagée avec d'autres calculs : ordres de grandeur, pas des mesures fines.

## C. Faut-il donner `time_cycles` aux réseaux ?

En Phase 3, `time_cycles` (l'âge du moteur) a nettement amélioré XGBoost (RMSE CV 18,64 → 16,86, test
d'ablation). Pour un réseau, la question se repose : il voit 30 cycles, mais pas depuis combien de temps le
moteur vole. On ajoute éventuellement une 16e entrée, `time_cycles` / 100 (constante fixe, rien n'est appris
sur les données, donc aucune fuite ; valeurs de 0 à 3,6 sur FD001).

**Décision prise sur une seule séparation de validation du train, jamais sur le test**, avec une règle écrite
avant l'expérience (dans le script, `TEXTE_REGLE_ENTREES`) :

In [ ]:
choix_entrees = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "03b_choix_entrees_fd001.csv"))
with open(os.path.join(DOSSIER_RESULTATS, "03b_choix_entrees_fd001.json"), encoding="utf-8") as fichier:
    decision_entrees = json.load(fichier)
print(decision_entrees["regle"])
print()

resume_choix = choix_entrees.groupby(["type_modele", "avec_time_cycles"])["rmse_decision"].agg(["mean", "std", "min", "max"])
resume_choix.columns = ["rmse_moyen_3_graines", "ecart_type", "min", "max"]
print(resume_choix.round(2).to_string())
print()
print(f"Moyenne sur les deux reseaux : avec time_cycles {decision_entrees['rmse_moyen_avec_time_cycles']:.2f}, "
      f"sans {decision_entrees['rmse_moyen_sans_time_cycles']:.2f} -> time_cycles garde : "
      f"{decision_entrees['garder_time_cycles']}")

AVEC_TIME_CYCLES = bool(decision_entrees["garder_time_cycles"])
colonnes_entree = lister_colonnes_entree(capteurs, AVEC_TIME_CYCLES)  # entrees de tous les reseaux qui suivent
print(f"Entrees retenues ({len(colonnes_entree)}) : {colonnes_entree}")

**`time_cycles` est gardé : les réseaux reçoivent 16 entrées** (15 capteurs + `time_cycles` / 100).

- RMSE sur les 20 moteurs de décision, moyenne de 3 graines : LSTM 12,09 avec contre 13,58 sans ; CNN 1D 13,88 avec
  contre 16,39 sans. Pour chaque réseau, le **moins bon** essai avec `time_cycles` (LSTM 12,50 ; CNN 14,41) fait
  mieux que le **meilleur** essai sans (13,34 ; 15,56) : l'écart n'est pas un effet de graine.
- Même conclusion qu'en Phase 3 pour XGBoost : l'âge du moteur aide, même quand le modèle voit 30 cycles d'historique.
- Limite : une seule séparation et 20 moteurs de décision. C'est un choix fait proprement sans le test, pas une preuve
  générale ; il est appliqué tel quel à FD002–FD004.

## D. Validation croisée sur FD001, mêmes plis que les arbres

### D.1 Protocole

Pour chacun des 5 plis (les plis de la Phase 4) et pour chaque réseau :

1. les 80 moteurs d'entraînement du pli sont séparés (graine 42) en **64 moteurs d'apprentissage** et
   **16 moteurs d'arrêt précoce** ; les 20 moteurs de validation du pli restent intouchés ;
2. un réseau **neuf** (graine 42) apprend sur les 64 moteurs, l'arrêt précoce surveille les 16 ;
3. il prédit les 20 moteurs de validation, à **chaque cycle** : prédictions hors-pli (OOF), une par ligne.

Les 10 apprentissages (2 réseaux × 5 plis) tournent en parallèle dans le script (1 cœur chacun, résultats
identiques à une exécution l'une après l'autre : vérifié par `tests/test_deep_learning.py`).

On relit les résultats et on vérifie leur cohérence : prédictions OOF alignées sur les lignes du train, et le
RMSE de chaque pli recalculé ici, à partir des plis de ce notebook, égal à celui du script (ce qui prouve que le
script a bien utilisé les mêmes plis).

In [ ]:
cv_reseaux = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "03b_cv_fd001.csv")).set_index("modele")
oof = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "03b_oof_fd001.csv.gz"))

# alignement ligne a ligne avec le train de ce notebook
assert np.array_equal(oof["unit_number"].to_numpy(), train["unit_number"].to_numpy())
assert np.array_equal(oof["time_cycles"].to_numpy(), train["time_cycles"].to_numpy())
assert np.array_equal(oof["RUL_cappee"].to_numpy(dtype=float), y.to_numpy(dtype=float))

for nom_reseau in cv_reseaux.index:  # chaque reseau (et la variante asymetrique, section E)
    rmse_par_pli_script = json.loads(cv_reseaux.loc[nom_reseau, "rmse_par_pli"])
    for position in range(len(plis)):
        idx_val = plis[position][1]
        rmse_recalcule = rmse(y.iloc[idx_val], oof[f"oof_{nom_reseau}"].iloc[idx_val])
        # tolerance : les predictions OOF sont ecrites avec 4 decimales
        assert abs(rmse_recalcule - rmse_par_pli_script[position]) < 1e-3, (nom_reseau, position)
    print(f"{nom_reseau:18s} : RMSE par pli {np.round(rmse_par_pli_script, 2)} (recalcules ici : identiques), "
          f"meilleures epoques {cv_reseaux.loc[nom_reseau, 'meilleures_epoques']}")

### D.2 Contrôle des références XGBoost

Les deux XGBoost ont été réentraînés en A.5 sur les mêmes plis. Contrôle : on doit retrouver exactement les chiffres
de `docs/resultats/04_comparaison_cv.csv` (RMSE CV 16,87 et 17,28).

In [ ]:
reference_04 = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "04_comparaison_cv.csv")).set_index("modele")
assert abs(resultats_xgb["XGBoost Phase 3"]["rmse_moyen"] - reference_04.loc["XGBoost Phase 3", "rmse_cv_moyen"]) < 1e-6
assert abs(resultats_xgb["XGBoost retenu"]["rmse_moyen"] - reference_04.loc["Optuna retenu (xgb_quadratique n°17)", "rmse_cv_moyen"]) < 1e-6
print("Chiffres de validation croisee de la Phase 4 retrouves a l'identique :")
for nom_modele in resultats_xgb:
    print(f"  {nom_modele} : RMSE CV moyen {resultats_xgb[nom_modele]['rmse_moyen']:.3f}")

### D.3 Tableau comparatif (validation croisée, mêmes 5 plis)

Mêmes colonnes que la Phase 4. Le temps d'entraînement des réseaux est la somme des 5 plis, chacun mesuré dans
son processus avec **1 cœur** (machine partagée avec d'autres calculs : ordre de grandeur, pas une mesure fine) ;
XGBoost est mesuré ici avec ses réglages par défaut (tous les cœurs). Prédiction : temps pour prédire toute la
trajectoire des moteurs de validation, ramené à un moteur (environ 206 cycles).

D'abord une petite fonction qui met une ligne de résultats au même format pour les arbres et pour les réseaux.

In [ ]:
def ligne_cv(nom, rmse_moyen, rmse_ecart_type, mae_moyen, nasa_moyen, surestimes, alertes_manquees, fausses_alertes,
             biais, temps_entrainement, prediction_ms):
    '''
    Construit une ligne du tableau de validation croisee (memes colonnes pour arbres et reseaux).

    Parameters
    ----------
    nom : str
        Nom du modele.
    rmse_moyen, rmse_ecart_type, mae_moyen, nasa_moyen : float
        Moyennes (et ecart-type du RMSE) sur les 5 plis.
    surestimes, alertes_manquees, fausses_alertes : int
        Comptes sur les 20 631 predictions hors-pli.
    biais : float
        Moyenne de (prediction - verite) hors-pli.
    temps_entrainement : float
        Entrainement cumule des 5 plis (s).
    prediction_ms : float
        Prediction d'une trajectoire de moteur (ms).

    Returns
    -------
    dict
        La ligne.
    '''
    ligne = {}
    ligne["modele"] = nom
    ligne["rmse_cv_moyen"] = rmse_moyen
    ligne["rmse_cv_ecart_type"] = rmse_ecart_type
    ligne["mae_cv_moyen"] = mae_moyen
    ligne["nasa_cv_moyen"] = nasa_moyen
    ligne["surestimes_oof_20"] = surestimes
    ligne["alertes_manquees_oof"] = alertes_manquees
    ligne["fausses_alertes_oof"] = fausses_alertes
    ligne["biais_oof"] = biais
    ligne["temps_entrainement_5_plis_s"] = temps_entrainement
    ligne["prediction_ms_par_moteur"] = prediction_ms
    return ligne

Le tableau : les deux réseaux (lus dans `03b_cv_fd001.csv`) et les deux XGBoost (calculés en D.2).

In [ ]:
NOMS_AFFICHES = {"lstm": "LSTM", "cnn1d": "CNN 1D"}
lignes = []
for nom_reseau in ["lstm", "cnn1d"]:
    r = cv_reseaux.loc[nom_reseau]
    lignes.append(ligne_cv(NOMS_AFFICHES[nom_reseau], r["rmse_cv_moyen"], r["rmse_cv_ecart_type"], r["mae_cv_moyen"],
                           r["nasa_cv_moyen"], r["surestimes_oof"], r["alertes_manquees_oof"], r["fausses_alertes_oof"],
                           r["biais_oof"], r["temps_entrainement_5_plis_s"], r["prediction_ms_par_moteur"]))
for nom_modele in resultats_xgb:
    r = resultats_xgb[nom_modele]
    lignes.append(ligne_cv(nom_modele, r["rmse_moyen"], r["rmse_ecart_type"], r["mae_moyen"], r["nasa_moyen"],
                           r["surestimes_oof"], r["alertes_manquees_oof"], r["fausses_alertes_oof"], r["biais_oof"],
                           r["temps_entrainement_total"], 1000.0 * r["temps_prediction_total"] / NB_MOTEURS_TRAIN))
tableau_cv = pd.DataFrame(lignes).set_index("modele")
tableau_cv.round(2)

**Un point d'équité à vérifier : le clip à 0.** Les prédictions des réseaux sont ramenées à 0 si elles sont
négatives ; les prédictions hors-pli des arbres ne le sont pas (protocole de la Phase 3). Le clip ne peut que
réduire l'erreur (la vérité est ≥ 0). On mesure son effet sur XGBoost : s'il est négligeable, la comparaison
du tableau est équitable.

In [ ]:
for nom_modele in resultats_xgb:
    predictions_oof_xgb = resultats_xgb[nom_modele]["predictions_oof"]
    predictions_clippees = np.clip(predictions_oof_xgb, a_min=0.0, a_max=None)
    rmse_par_pli_clippe = []
    for idx_train, idx_val in plis:
        rmse_par_pli_clippe.append(rmse(y.iloc[idx_val], predictions_clippees[idx_val]))
    nb_negatives = int(np.sum(predictions_oof_xgb < 0))
    print(f"{nom_modele} : {nb_negatives} predictions OOF negatives ; RMSE CV moyen {resultats_xgb[nom_modele]['rmse_moyen']:.3f} "
          f"-> {np.mean(rmse_par_pli_clippe):.3f} avec clip a 0")

### D.4 Les écarts sont-ils significatifs ? Tests appariés par moteur

Comme en Phase 4 : pour chaque modèle, le **RMSE hors-pli de chacun des 100 moteurs**, puis comparaison moteur
par moteur (`comparer_apparie` : test t apparié et Wilcoxon signé). Trois comparaisons fixées à l'avance :
LSTM contre XGBoost Phase 3, CNN 1D contre XGBoost Phase 3, LSTM contre CNN 1D. Seuil de Bonferroni :
0,05 / 3 ≈ 0,017 pour les deux tests. Différence moyenne négative = le premier modèle a un RMSE par moteur plus
petit (il est meilleur).

In [ ]:
rmse_par_moteur = {}  # nom -> RMSE hors-pli de chacun des 100 moteurs (tries par numero)
rmse_par_moteur["LSTM"] = erreurs_par_moteur(y, oof["oof_lstm"], groups)["rmse"].to_numpy()
rmse_par_moteur["CNN 1D"] = erreurs_par_moteur(y, oof["oof_cnn1d"], groups)["rmse"].to_numpy()
for nom_modele in resultats_xgb:
    rmse_par_moteur[nom_modele] = erreurs_par_moteur(y, resultats_xgb[nom_modele]["predictions_oof"], groups)["rmse"].to_numpy()

SEUIL_BONFERRONI_CV = 0.05 / 3
comparaisons_cv = [("LSTM", "XGBoost Phase 3"), ("CNN 1D", "XGBoost Phase 3"), ("LSTM", "CNN 1D")]
lignes = []
for nom_a, nom_b in comparaisons_cv:
    resultat = comparer_apparie(rmse_par_moteur[nom_a], rmse_par_moteur[nom_b])
    ligne = {}
    ligne["comparaison"] = f"{nom_a} vs {nom_b}"
    ligne["rmse_moyen_par_moteur_a"] = float(np.mean(rmse_par_moteur[nom_a]))
    ligne["rmse_moyen_par_moteur_b"] = float(np.mean(rmse_par_moteur[nom_b]))
    ligne["difference_moyenne"] = resultat["difference_moyenne"]
    ligne["moteurs_ou_a_est_meilleur"] = int(np.sum(rmse_par_moteur[nom_a] < rmse_par_moteur[nom_b]))
    ligne["p_valeur_t"] = resultat["p_valeur_t"]
    ligne["p_valeur_wilcoxon"] = resultat["p_valeur_wilcoxon"]
    ligne["significatif_bonferroni"] = bool(resultat["p_valeur_t"] < SEUIL_BONFERRONI_CV and resultat["p_valeur_wilcoxon"] < SEUIL_BONFERRONI_CV)
    lignes.append(ligne)
tableau_tests_cv = pd.DataFrame(lignes).set_index("comparaison")
tableau_tests_cv.round(4)

### D.5 Courbes d'apprentissage

Pour chaque pli : l'erreur sur les moteurs d'**arrêt précoce** après chaque époque (trait plein, RMSE en cycles)
et l'erreur d'**entraînement** (pointillés, √perte × 125 : RMSE équivalent en cycles, calculé pendant l'époque
avec le dropout actif, donc un peu pessimiste). Le point marque l'époque gardée. Lecture attendue : les deux
courbes descendent vite, puis la courbe d'arrêt précoce se stabilise (ou remonte) pendant que celle
d'entraînement continue de baisser : c'est là que l'arrêt précoce intervient.

In [ ]:
with open(os.path.join(DOSSIER_RESULTATS, "03b_historiques_fd001.json"), encoding="utf-8") as fichier:
    historiques = json.load(fichier)

couleurs_plis = ["#2a78d6", "#eb6834", "#1baf7a", "#9b59b6", "#52514e"]  # une couleur par pli
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True)
for ax, nom_reseau in zip(axes, ["lstm", "cnn1d"]):
    plis_historique = historiques[nom_reseau]["plis"]
    for numero in range(len(plis_historique)):
        historique = plis_historique[numero]
        epoques = np.arange(1, len(historique["rmse_validation"]) + 1)
        rmse_entrainement = np.sqrt(np.array(historique["perte_entrainement"])) * ECHELLE_CIBLE
        ax.plot(epoques, historique["rmse_validation"], color=couleurs_plis[numero], linewidth=1.6,
                label=f"pli {numero + 1} : arrêt précoce (époque gardée {historique['meilleure_epoque']})")
        ax.plot(epoques, rmse_entrainement, color=couleurs_plis[numero], linewidth=1, linestyle=":")
        meilleure = historique["meilleure_epoque"]
        ax.scatter([meilleure], [historique["rmse_validation"][meilleure - 1]], color=couleurs_plis[numero], s=40, zorder=3)
    ax.set_title(f"{NOMS_AFFICHES[nom_reseau]} — trait plein : moteurs d'arrêt précoce ; pointillés : entraînement")
    ax.set_xlabel("époque")
    ax.set_ylim(8, 40)
    ax.legend(fontsize=8)
axes[0].set_ylabel("RMSE (cycles)")
fig.suptitle("Courbes d'apprentissage FD001, validation croisée 5 plis (mêmes plis que les arbres)")
plt.tight_layout()
plt.savefig(os.path.join(DOSSIER_FIGURES, "03b_courbes_apprentissage.png"), dpi=150)
plt.show()

**Figure de synthèse (validation croisée)** : un critère par panneau. Panneau RMSE : barre = moyenne des 5 plis,
points = les 5 plis.

In [ ]:
noms_figure = ["XGBoost Phase 3", "XGBoost retenu", "LSTM", "CNN 1D"]
couleurs_figure = [COULEUR_1, COULEUR_4, COULEUR_2, COULEUR_3]
rmse_plis_figure = {}
rmse_plis_figure["XGBoost Phase 3"] = resultats_xgb["XGBoost Phase 3"]["rmse_par_pli"]
rmse_plis_figure["XGBoost retenu"] = resultats_xgb["XGBoost retenu"]["rmse_par_pli"]
rmse_plis_figure["LSTM"] = json.loads(cv_reseaux.loc["lstm", "rmse_par_pli"])
rmse_plis_figure["CNN 1D"] = json.loads(cv_reseaux.loc["cnn1d", "rmse_par_pli"])

fig, axes = plt.subplots(1, 4, figsize=(17, 4.5))
panneaux = [("rmse_cv_moyen", "RMSE CV moyen (cycles)"), ("nasa_cv_moyen", "score NASA CV moyen"),
            ("surestimes_oof_20", "surestimés > 20 cycles (lignes OOF)"), ("temps_entrainement_5_plis_s", "entraînement 5 plis (s, échelle log)")]
for ax, (colonne, titre) in zip(axes, panneaux):
    for position in range(len(noms_figure)):
        nom_modele = noms_figure[position]
        valeur = tableau_cv.loc[nom_modele, colonne]
        ax.bar(position, valeur, color=couleurs_figure[position], width=0.65)
        hauteur_texte = valeur  # le texte se place juste au-dessus de la barre...
        texte = f"{valeur:,.0f}".replace(",", " ")
        if colonne == "rmse_cv_moyen":
            ax.scatter(np.full(5, position), rmse_plis_figure[nom_modele], color="black", s=14, zorder=3)
            hauteur_texte = max(rmse_plis_figure[nom_modele]) + 0.3  # ... ou au-dessus du pli le plus haut
            texte = f"{valeur:.2f}"
        ax.text(position, hauteur_texte, texte, ha="center", va="bottom", fontsize=8)
    ax.set_xticks(np.arange(len(noms_figure)))
    ax.set_xticklabels(noms_figure, rotation=20, fontsize=8)
    ax.set_title(titre, fontsize=10)
    if colonne == "temps_entrainement_5_plis_s":
        ax.set_yscale("log")
fig.suptitle("FD001 — validation croisée, mêmes 5 plis : arbres contre réseaux (plus bas = mieux)")
plt.tight_layout()
plt.savefig(os.path.join(DOSSIER_FIGURES, "03b_comparaison_cv.png"), dpi=150)
plt.show()

### D.6 Conclusion de la section D

- **Le LSTM fait nettement mieux que XGBoost en validation croisée.** RMSE CV 13,70 ± 1,76, contre 16,87 ± 1,25
  (XGBoost Phase 3) et 17,28 ± 2,10 (XGBoost retenu). Test apparié sur les 100 moteurs : RMSE par moteur 12,96
  contre 15,51 en moyenne (−2,55 cycles), plus petit sur 61 moteurs sur 100, p < 0,0001 (t) et p = 0,0001
  (Wilcoxon) : significatif après correction de Bonferroni. L'écart des moyennes (3,2 cycles) dépasse la variation
  d'un pli à l'autre (écart-type de 1,3 à 1,8).
- **Il est aussi plus prudent, sans avoir été réglé pour** : score NASA CV 15 630 (contre 29 783 et 21 075) ;
  1 503 lignes surestimées de plus de 20 cycles (contre 2 384 et 1 517) ; 298 alertes manquées (contre 552 et 492) ;
  173 fausses alertes (contre 259 et 381) ; biais −0,26 cycle.
- **Le CNN 1D est au niveau de XGBoost Phase 3, pas au-dessus** : RMSE CV 16,96 contre 16,87, différence par
  moteur +0,09 (p = 0,89 / 0,84). Il est moins bon que le LSTM (−2,64 cycles par moteur en faveur du LSTM,
  p < 0,0001). Ses courbes d'apprentissage sont très irrégulières (sauts jusqu'à 29 cycles d'une époque à l'autre)
  et l'arrêt précoce garde des époques précoces (9 à 14).
- **Le clip à 0 n'explique rien** : XGBoost Phase 3 n'a que 25 prédictions hors-pli négatives sur 20 631, et son
  RMSE CV ne bouge pas au millième près avec le clip (16,867).
- **Équité des données** : dans chaque pli, les réseaux apprennent sur 64 moteurs (16 sont réservés à l'arrêt
  précoce), XGBoost sur 80. L'avantage du LSTM ne vient donc pas de plus de données.
- **Coût** : plus de 100 fois plus de temps d'entraînement que XGBoost Phase 3 (385 s pour les 5 plis, contre 2 à
  4 s selon les exécutions et la charge de la machine ; 1 cœur pour le réseau, tous les cœurs pour XGBoost) ;
  prédiction de la trajectoire d'un moteur en 11 ms (LSTM) et 57 ms (CNN), contre environ 0,1 ms.
- **Pourquoi le CNN fait-il moins bien que le LSTM ? Hypothèses, non testées** : la moyenne globale sur les 30
  positions efface *où* un motif apparaît dans la fenêtre, alors que les derniers cycles comptent le plus (le LSTM
  sort l'état du dernier cycle) ; et un apprentissage instable (courbes irrégulières). Les tester demanderait de
  modifier l'architecture après avoir vu ces résultats, à juger sur la validation croisée seulement.

## E. Variante : perte asymétrique (α = 2,26) pour le meilleur réseau

En Phase 4, pénaliser davantage les surestimations **dans l'apprentissage** a rendu XGBoost plus prudent
(perte quadratique asymétrique : erreur² × α si le modèle surestime, × 1 sinon ; α = 2,26 retenu par Optuna).
Même idée pour le réseau qui a le plus petit RMSE CV en section D, avec la même valeur de α (non réoptimisée pour
le réseau) et le même protocole (mêmes plis, même arrêt précoce, qui surveille alors la perte asymétrique).
Tests appariés par moteur contre le même réseau à α = 1 et contre XGBoost retenu, sur trois métriques par
moteur : RMSE, pénalité NASA (somme des pénalités des cycles du moteur), nombre de cycles surestimés de plus de 20.

D'abord la fonction qui calcule ces trois métriques moteur par moteur (même définition qu'en Phase 4) :

In [ ]:
def metriques_par_moteur(y_vrai, y_predit, groupes):
    '''
    Calcule, pour chaque moteur, le RMSE, la penalite NASA totale et le nombre de cycles surestimes de plus de 20.

    Parameters
    ----------
    y_vrai, y_predit : array-like
        Verites et predictions, une par ligne.
    groupes : array-like
        Numero de moteur de chaque ligne.

    Returns
    -------
    pandas.DataFrame
        Une ligne par moteur (tries par numero) : unit_number, rmse, nasa, surestimes.
    '''
    vrais = np.asarray(y_vrai, dtype=float)
    predits = np.asarray(y_predit, dtype=float)
    numeros = np.asarray(groupes)
    lignes_moteurs = []
    for moteur_id in np.unique(numeros):
        masque = numeros == moteur_id
        ligne = {}
        ligne["unit_number"] = moteur_id
        ligne["rmse"] = rmse(vrais[masque], predits[masque])
        ligne["nasa"] = score_nasa(vrais[masque], predits[masque])
        ligne["surestimes"] = int(np.sum(predits[masque] - vrais[masque] > SEUIL_SURESTIMATION))
        lignes_moteurs.append(ligne)
    return pd.DataFrame(lignes_moteurs)

Le tableau de validation croisée : le réseau avec α = 1 (section D), le même réseau avec α = 2,26, et les deux
XGBoost pour repère.

In [ ]:
noms_asymetriques = []
for nom_reseau in cv_reseaux.index:
    if float(cv_reseaux.loc[nom_reseau, "alpha"]) != 1.0:
        noms_asymetriques.append(nom_reseau)

if len(noms_asymetriques) == 0:
    print("Variante asymetrique non calculee (etape 'asymetrique' du script non executee).")
else:
    nom_asym = noms_asymetriques[0]
    reseau_base = str(cv_reseaux.loc[nom_asym, "type_modele"])
    alpha_asym = float(cv_reseaux.loc[nom_asym, "alpha"])
    r = cv_reseaux.loc[nom_asym]
    lignes = [tableau_cv.loc[NOMS_AFFICHES[reseau_base]].to_dict()]
    lignes[0]["modele"] = f"{NOMS_AFFICHES[reseau_base]} (alpha = 1)"
    ligne_asym = ligne_cv(f"{NOMS_AFFICHES[reseau_base]} (alpha = {alpha_asym:.2f})", r["rmse_cv_moyen"], r["rmse_cv_ecart_type"],
                          r["mae_cv_moyen"], r["nasa_cv_moyen"], r["surestimes_oof"], r["alertes_manquees_oof"],
                          r["fausses_alertes_oof"], r["biais_oof"], r["temps_entrainement_5_plis_s"], r["prediction_ms_par_moteur"])
    lignes.append(ligne_asym)
    for nom_modele in ["XGBoost Phase 3", "XGBoost retenu"]:
        ligne = tableau_cv.loc[nom_modele].to_dict()
        ligne["modele"] = nom_modele
        lignes.append(ligne)
    print(pd.DataFrame(lignes).set_index("modele").round(2).to_string())

Tests appariés par moteur (100 moteurs) : différence moyenne négative = la variante asymétrique a une valeur plus
petite (meilleure) que la référence.

In [ ]:
if len(noms_asymetriques) == 0:
    print("Variante asymetrique non calculee.")
else:
    # tests apparies par moteur : variante asymetrique contre le meme reseau (alpha = 1) et contre XGBoost retenu
    par_moteur_asym = metriques_par_moteur(y, oof[f"oof_{nom_asym}"], groups)
    references = {}
    references[f"{NOMS_AFFICHES[reseau_base]} (alpha = 1)"] = metriques_par_moteur(y, oof[f"oof_{reseau_base}"], groups)
    references["XGBoost retenu"] = metriques_par_moteur(y, resultats_xgb["XGBoost retenu"]["predictions_oof"], groups)
    lignes = []
    for nom_reference in references:
        for metrique in ["rmse", "nasa", "surestimes"]:
            valeurs_asym = par_moteur_asym[metrique].to_numpy(dtype=float)
            valeurs_reference = references[nom_reference][metrique].to_numpy(dtype=float)
            resultat = comparer_apparie(valeurs_asym, valeurs_reference)
            ligne = {}
            ligne["variante asymetrique vs"] = nom_reference
            ligne["metrique_par_moteur"] = metrique
            ligne["moyenne_variante"] = float(np.mean(valeurs_asym))
            ligne["moyenne_reference"] = float(np.mean(valeurs_reference))
            ligne["difference_moyenne"] = resultat["difference_moyenne"]
            ligne["p_valeur_t"] = resultat["p_valeur_t"]
            ligne["p_valeur_wilcoxon"] = resultat["p_valeur_wilcoxon"]
            lignes.append(ligne)
    print(pd.DataFrame(lignes).set_index(["variante asymetrique vs", "metrique_par_moteur"]).round(4).to_string())

### E.1 Conclusion de la section E

- **La perte asymétrique rend le LSTM plus prudent, sans perte de précision détectable.** RMSE CV 13,87 contre 13,70
  (RMSE par moteur : −0,08, p = 0,78 / 0,38) ; lignes surestimées de plus de 20 cycles 930 contre 1 503 (−38 % ;
  par moteur 9,3 contre 15,0, p < 0,0001) ; score NASA CV 12 286 contre 15 630 (−21 % ; pénalité par moteur
  p = 0,008 / 0,002) ; alertes manquées 215 contre 298.
- **Le prix** : 317 fausses alertes contre 173 (+83 %) et un biais de −3,98 cycles contre −0,26 (le modèle
  sous-estime en moyenne de 4 cycles). C'est le même compromis qu'en Phase 4 pour XGBoost.
- **Contre XGBoost retenu** (même α, même idée) : RMSE par moteur plus petit (12,88 contre 15,34, p < 0,0001),
  pénalité NASA par moteur plus petite (614 contre 1 054, p < 0,0001), moins de cycles surestimés par moteur (9,3
  contre 15,2, p = 0,002 / 0,010), et moins de fausses alertes (317 contre 381).
- **Limites** : α = 2,26 a été choisi pour XGBoost, pas réoptimisé pour le réseau ; la variante n'est évaluée qu'en
  validation croisée, **pas sur le test**, et ce n'est **pas** le modèle publié (fichiers de prédictions : LSTM
  avec α = 1). Choisir α = 1 ou α = 2,26 pour le LSTM final est une décision d'équipe, à fixer par une règle écrite
  avant de regarder le test, comme en Phase 4.

## F. Modèle final et jeu de test officiel FD001 (évalué une seule fois)

### F.1 Protocole

- **Données** : les 100 moteurs d'entraînement, sans moteurs d'arrêt précoce (il n'y a plus de validation).
- **Nombre d'époques** : fixé à l'avance par la validation croisée, = la **médiane des 5 meilleures époques**
  des plis (jamais choisi sur le test). Limite assumée : avec 100 moteurs au lieu de 64, une époque contient
  environ 1,5 fois plus de lots, donc un peu plus de mises à jour qu'en validation croisée.
- **Test** : prédiction à **chaque cycle** de chaque moteur test (fichiers de prédictions pour le tableau de
  bord), puis métriques au **dernier cycle observé**, contre la RUL brute de `RUL_FD001.txt`, exactement comme
  `evaluer_sur_test` (Phase 3).
- **Robustesse** : le modèle final est entraîné avec 3 graines (42, 43, 44). Le modèle publié (fichiers de
  prédictions, poids `models/FD001__lstm.pt` et `models/FD001__cnn1d.pt`) est celui de la graine 42.

In [ ]:
test_reseaux = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "03b_test_fd001.csv"))
predictions_dernier_cycle = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "03b_predictions_test_fd001.csv"))

# alignement : memes moteurs, dans le meme ordre que y_test_vraie ; meme RUL vraie
assert np.array_equal(predictions_dernier_cycle["unit_number"].to_numpy(), moteurs_test)
assert np.array_equal(predictions_dernier_cycle["rul_vraie"].to_numpy(dtype=float), y_test_vraie.astype(float))

for nom_reseau in ["lstm", "cnn1d"]:
    epoques_plis = json.loads(cv_reseaux.loc[nom_reseau, "meilleures_epoques"])
    nb_epoques_final = int(test_reseaux[test_reseaux["modele"] == nom_reseau]["nb_epoques"].iloc[0])
    print(f"{NOMS_AFFICHES[nom_reseau]} : meilleures epoques des 5 plis {epoques_plis} -> mediane = {nb_epoques_final} "
          f"epoques pour le modele final")

### F.2 Contrôle des références XGBoost sur le test

Calculées en A.5. XGBoost Phase 3 doit redonner RMSE 19,70 et score NASA 852,9 ; XGBoost retenu 19,73 et 675,4.

In [ ]:
assert round(resultats_test_xgb["XGBoost Phase 3"]["rmse"], 2) == 19.70
assert round(resultats_test_xgb["XGBoost Phase 3"]["score_nasa"], 1) == 852.9
assert round(resultats_test_xgb["XGBoost retenu"]["rmse"], 2) == 19.73
assert round(resultats_test_xgb["XGBoost retenu"]["score_nasa"], 1) == 675.4
print("Chiffres de test des Phases 3 et 4 retrouves.")

### F.3 Tableau du test officiel (graine 42 pour les réseaux)

In [ ]:
lignes = []
for nom_reseau in ["lstm", "cnn1d"]:
    r = test_reseaux[(test_reseaux["modele"] == nom_reseau) & (test_reseaux["graine"] == 42)].iloc[0]
    ligne = {}
    ligne["modele"] = NOMS_AFFICHES[nom_reseau]
    ligne["rmse_test"] = r["rmse"]
    ligne["rmse_test_rul_plafonnee"] = r["rmse_contre_rul_plafonnee"]
    ligne["mae_test"] = r["mae"]
    ligne["nasa_test"] = r["score_nasa"]
    ligne["surestimes_test_20"] = int(r["nb_surestimes"])
    ligne["alertes_manquees_test"] = int(r["alertes_manquees"])
    ligne["fausses_alertes_test"] = int(r["fausses_alertes"])
    ligne["entrainement_final_s"] = r["duree_entrainement_s"]
    ligne["prediction_ms_par_moteur_test"] = r["prediction_ms_par_moteur_test"]
    lignes.append(ligne)
for nom_modele in resultats_test_xgb:
    r = resultats_test_xgb[nom_modele]
    ligne = {}
    ligne["modele"] = nom_modele
    ligne["rmse_test"] = r["rmse"]
    ligne["rmse_test_rul_plafonnee"] = r["rmse_contre_rul_plafonnee"]
    ligne["mae_test"] = r["mae"]
    ligne["nasa_test"] = r["score_nasa"]
    ligne["surestimes_test_20"] = r["nb_surestimes"]
    ligne["alertes_manquees_test"] = r["alertes_manquees"]
    ligne["fausses_alertes_test"] = r["fausses_alertes"]
    ligne["entrainement_final_s"] = np.nan  # mesure en Phase 4 (moins d'une seconde)
    ligne["prediction_ms_par_moteur_test"] = temps_prediction_xgb_ms[nom_modele]
    lignes.append(ligne)
tableau_test = pd.DataFrame(lignes).set_index("modele")
tableau_test.round(2)

### F.4 Robustesse à la graine (3 graines)

Un bon chiffre peut venir d'une graine chanceuse. Pour un réseau, la graine change les poids de départ, l'ordre
des lots et le dropout : on s'attend à une vraie variabilité (contrairement à XGBoost Phase 3, identique pour
toutes les graines en Phase 4). Moyenne ± écart-type sur les graines 42, 43 et 44.

In [ ]:
colonnes_graines = ["rmse", "rmse_contre_rul_plafonnee", "mae", "score_nasa", "nb_surestimes", "alertes_manquees", "fausses_alertes"]
resume_graines = test_reseaux.groupby("modele")[colonnes_graines].agg(["mean", "std"])
print(resume_graines.round(2).to_string())
print()
print("Pour comparaison (Phase 4, 5 graines) : XGBoost retenu RMSE test 19,65 +/- 0,05, score NASA 666 +/- 6 ; "
      "XGBoost Phase 3 identique pour toutes les graines (19,70 / 852,9).")

### F.5 Tests appariés sur les 100 moteurs test

Erreur absolue de chaque moteur test (graine 42 pour les réseaux), comparée moteur par moteur. Cinq comparaisons
(chaque réseau contre les deux XGBoost, et LSTM contre CNN) : seuil de Bonferroni 0,05 / 5 = 0,01. Avec 100
points seulement, il faut un écart net pour conclure.

In [ ]:
erreurs_absolues_test = {}
erreurs_absolues_test["LSTM"] = np.abs(predictions_dernier_cycle["lstm_graine42"].to_numpy() - y_test_vraie)
erreurs_absolues_test["CNN 1D"] = np.abs(predictions_dernier_cycle["cnn1d_graine42"].to_numpy() - y_test_vraie)
for nom_modele in resultats_test_xgb:
    erreurs_absolues_test[nom_modele] = np.abs(resultats_test_xgb[nom_modele]["predictions"] - y_test_vraie)

SEUIL_BONFERRONI_TEST = 0.05 / 5
comparaisons_test = [("LSTM", "XGBoost Phase 3"), ("CNN 1D", "XGBoost Phase 3"), ("LSTM", "XGBoost retenu"),
                     ("CNN 1D", "XGBoost retenu"), ("LSTM", "CNN 1D")]
lignes = []
for nom_a, nom_b in comparaisons_test:
    resultat = comparer_apparie(erreurs_absolues_test[nom_a], erreurs_absolues_test[nom_b])
    ligne = {}
    ligne["comparaison"] = f"{nom_a} vs {nom_b}"
    ligne["erreur_absolue_moyenne_a"] = float(np.mean(erreurs_absolues_test[nom_a]))
    ligne["erreur_absolue_moyenne_b"] = float(np.mean(erreurs_absolues_test[nom_b]))
    ligne["difference_moyenne"] = resultat["difference_moyenne"]
    ligne["moteurs_ou_a_est_meilleur"] = int(np.sum(erreurs_absolues_test[nom_a] < erreurs_absolues_test[nom_b]))
    ligne["p_valeur_t"] = resultat["p_valeur_t"]
    ligne["p_valeur_wilcoxon"] = resultat["p_valeur_wilcoxon"]
    ligne["significatif_bonferroni"] = bool(resultat["p_valeur_t"] < SEUIL_BONFERRONI_TEST and resultat["p_valeur_wilcoxon"] < SEUIL_BONFERRONI_TEST)
    lignes.append(ligne)
pd.DataFrame(lignes).set_index("comparaison").round(4)

### F.6 Figure : prédictions contre vraies RUL sur le test

Même lecture qu'en Phases 3 et 4 : diagonale = prédiction parfaite ; au-dessus = surestimation (dangereux) ; les
moteurs surestimés de plus de 20 cycles sont en triangles orange. La ligne horizontale pointillée rappelle le
plafond de 125 : aucun modèle n'a appris à prédire au-delà, d'où l'erreur sur les moteurs de RUL vraie > 125.

In [ ]:
predictions_figure = {}
predictions_figure["XGBoost Phase 3"] = resultats_test_xgb["XGBoost Phase 3"]["predictions"]
predictions_figure["XGBoost retenu"] = resultats_test_xgb["XGBoost retenu"]["predictions"]
predictions_figure["LSTM"] = predictions_dernier_cycle["lstm_graine42"].to_numpy()
predictions_figure["CNN 1D"] = predictions_dernier_cycle["cnn1d_graine42"].to_numpy()

limite = float(np.max(y_test_vraie)) + 10
fig, axes = plt.subplots(2, 2, figsize=(12, 11), sharex=True, sharey=True)
for ax, nom_modele in zip(axes.flatten(), ["XGBoost Phase 3", "XGBoost retenu", "LSTM", "CNN 1D"]):
    predictions = predictions_figure[nom_modele]
    surestime = (predictions - y_test_vraie) > SEUIL_SURESTIMATION
    ax.scatter(y_test_vraie[~surestime], predictions[~surestime], color=COULEUR_1, alpha=0.6, s=28, label="moteurs test")
    ax.scatter(y_test_vraie[surestime], predictions[surestime], color=COULEUR_2, marker="^", s=60, alpha=0.9,
               label=f"surestimés > {SEUIL_SURESTIMATION} cycles ({int(surestime.sum())})")
    ax.plot([0, limite], [0, limite], linestyle="--", color="black", linewidth=1, label="prédiction = vérité")
    ax.plot([0, limite], [SEUIL_SURESTIMATION, limite + SEUIL_SURESTIMATION], linestyle=":", color=COULEUR_TEXTE,
            label=f"vérité + {SEUIL_SURESTIMATION}")
    ax.axhline(PLAFOND_RUL, color=COULEUR_TEXTE, linestyle="-.", linewidth=0.8)
    ax.set_title(f"{nom_modele}\nRMSE = {rmse(y_test_vraie, predictions):.2f}, score NASA = {score_nasa(y_test_vraie, predictions):.1f}, "
                 f"surestimés > 20 : {int(surestime.sum())}/100", fontsize=10)
    ax.set_xlim(0, limite)
    ax.set_ylim(0, limite)
    ax.legend(fontsize=8, loc="upper left")
for ax in axes[1, :]:
    ax.set_xlabel("RUL vraie (RUL_FD001.txt)")
for ax in axes[:, 0]:
    ax.set_ylabel("RUL prédite au dernier cycle observé")
fig.suptitle("Test officiel FD001 (100 moteurs, évalué une fois) — arbres et réseaux")
plt.tight_layout()
plt.savefig(os.path.join(DOSSIER_FIGURES, "03b_predictions_vs_vraies_rul_test.png"), dpi=150)
plt.show()

### F.7 Trajectoires de deux moteurs test (prédiction à chaque cycle)

Le réseau prédit à **chaque** cycle d'un moteur test (fenêtre des 30 derniers cycles observés) : c'est ce
qu'affichera le tableau de bord. On lit les fichiers de prédictions complets (`sorties/predictions/`, produits par
les scripts, non versionnés : la cellule s'adapte s'ils manquent). Moteurs choisis par une règle simple, sans
regarder les prédictions : les deux moteurs test qui ont le plus de cycles observés parmi ceux dont la RUL vraie
au dernier cycle est ≤ 30 (on voit ainsi la descente jusqu'à la zone d'alerte).

In [ ]:
fichiers_trajectoires = {}
for cle_modele, nom_modele in [("xgboost_phase3", "XGBoost Phase 3"), ("lstm", "LSTM"), ("cnn1d", "CNN 1D")]:
    try:
        df_predictions, _ = lire_predictions("FD001", cle_modele)
        fichiers_trajectoires[nom_modele] = df_predictions
    except (FileNotFoundError, ValueError):
        print(f"Fichier de predictions FD001 '{cle_modele}' absent : relancer les scripts pour cette figure.")

if len(fichiers_trajectoires) == 3:
    derniers_cycles = predictions_dernier_cycle[predictions_dernier_cycle["rul_vraie"] <= SEUIL_ALERTE]
    moteurs_choisis = derniers_cycles.sort_values("nb_cycles_observes", ascending=False)["unit_number"].to_numpy()[:2]
    couleurs_modeles = {"XGBoost Phase 3": COULEUR_1, "LSTM": COULEUR_2, "CNN 1D": COULEUR_3}
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
    for ax, moteur_id in zip(axes, moteurs_choisis):
        for nom_modele in fichiers_trajectoires:
            df_predictions = fichiers_trajectoires[nom_modele]
            df_moteur = df_predictions[df_predictions["unit_number"] == moteur_id]
            ax.plot(df_moteur["time_cycles"], df_moteur["rul_predite"], color=couleurs_modeles[nom_modele], linewidth=1.5,
                    label=nom_modele)
        ax.plot(df_moteur["time_cycles"], df_moteur["rul_vraie_cappee"], color="black", linestyle="--", linewidth=1.2,
                label="RUL vraie (plafonnée à 125)")
        ax.axhline(SEUIL_ALERTE, color=COULEUR_TEXTE, linestyle=":", label=f"seuil d'alerte ({SEUIL_ALERTE} cycles)")
        ax.set_title(f"Moteur test {moteur_id} ({len(df_moteur)} cycles observés)")
        ax.set_xlabel("cycle")
        ax.legend(fontsize=8)
    axes[0].set_ylabel("RUL prédite à chaque cycle")
    fig.suptitle("FD001 — RUL prédite cycle après cycle sur deux moteurs test")
    plt.tight_layout()
    plt.savefig(os.path.join(DOSSIER_FIGURES, "03b_trajectoires_moteurs_test.png"), dpi=150)
    plt.show()

### F.8 Conclusion de la section F

- **Sur le test officiel, le LSTM fait mieux que les deux XGBoost** : RMSE 13,86 contre 19,70 (Phase 3) et 19,73
  (retenu) ; score NASA 293,2 contre 852,9 et 675,4 ; MAE 10,36 contre 14,36 et 14,32 ; **0 alerte manquée**
  contre 7 et 7 ; 8 moteurs surestimés de plus de 20 cycles contre 9 et 9 ; 1 fausse alerte dans les trois cas.
  Test apparié sur l'erreur absolue des 100 moteurs : −3,99 cycles contre XGBoost Phase 3 (p = 0,0009 / 0,0054)
  et −3,95 contre XGBoost retenu (p = 0,0008 / 0,0024), significatifs au seuil de Bonferroni de 0,01. C'est cohérent
  avec la validation croisée (section D) : le résultat du test confirme, il ne surprend pas.
- **Les surestimations dangereuses de XGBoost disparaissent** : les moteurs 37, 18 et 41 (RUL vraie 21, 28 et 18),
  prédits à 75, 62 et 50 cycles par XGBoost Phase 3, le sont à 18,6 ; 21,4 et 12,6 par le LSTM. Ses 8 moteurs
  surestimés de plus de 20 cycles ont tous une RUL vraie entre 57 et 98 cycles (loin du seuil d'alerte), avec un
  écart de 20 à 40 cycles.
- **Robustesse** : sur 3 graines, LSTM RMSE 14,22 ± 0,30, score NASA 343 ± 43, alertes manquées 1 ± 1. La graine 42
  (13,86) est la meilleure des trois ; la moins bonne (14,40) reste loin des 19,70 de XGBoost.
- **Le CNN 1D n'est pas fiable** : RMSE 21,27 avec la graine 42 (moins bon que XGBoost), 17,64 et 17,28 avec les deux
  autres ; score NASA de 598 à 2 067 selon la graine (1 143 ± 805). Aucune différence significative avec XGBoost
  sur le test (p ≥ 0,14). C'est l'instabilité déjà visible en validation croisée.
- **Coût** : le LSTM prédit la trajectoire complète d'un moteur test en 7 ms (1 cœur), contre moins de 0,1 ms pour
  XGBoost — compatible avec la cible de 200 ms de la démo, mais plus de 100 fois plus lent ; entraînement final :
  81 s, contre moins d'une seconde.

## G. FD002, FD003 et FD004

### G.1 Ce qui change, et le protocole

- **Préparation** : `src/sous_jeux.py` (même pipeline que FD001 ; pour FD002 et FD004, six régimes de vol trouvés
  par K-Means sur les réglages, et une normalisation min-max **par régime**, apprise sur le train). Les réseaux
  reçoivent les capteurs normalisés puis lissés du sous-jeu (17 pour FD002 / FD004, 16 pour FD003), plus
  `time_cycles` / 100 si la section C l'a retenu. Mêmes architectures et mêmes réglages qu'en FD001 (non
  réoptimisés).
- **Validation des modèles publiés** : une **séparation unique** des moteurs du train (graine 42) : 80 % pour
  apprendre, 20 % pour l'arrêt précoce et le choix du nombre d'époques. Ces métriques de validation (moteurs
  d'arrêt précoce, donc un peu optimistes) **ne sont pas comparables** à la validation croisée des arbres.
- **Validation croisée complémentaire** (G.3) : les mêmes réseaux, en validation croisée 5 plis **avec les plis des
  arbres** de chaque sous-jeu (même protocole qu'en FD001). Elle n'a servi à aucun choix : elle permet de comparer
  réseaux et arbres en validation, sur les mêmes moteurs, en plus du test.
- **Modèle final** : tout le train, nombre d'époques = meilleure époque de la séparation ; prédiction à chaque
  cycle de chaque moteur test ; métriques au dernier cycle observé.
- **Lecture du RMSE test** : FD002 et FD004 contiennent des moteurs test dont la RUL vraie dépasse largement 125 ;
  aucun modèle ne prédit au-delà de 125 (cible plafonnée), d'où un RMSE contre la RUL brute très supérieur au RMSE
  contre la RUL plafonnée. On donne les deux, et le score NASA **par moteur** (les sous-jeux n'ont pas le même
  nombre de moteurs test).

On relit les résultats des réseaux (`03b_deep_learning.csv`) et ceux des arbres (`04b_sous_jeux_arbres.csv`,
produits par `scripts/generer_predictions_arbres.py`), puis une petite fonction les met au même format.

In [ ]:
resultats_dl = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "03b_deep_learning.csv"))
chemin_arbres = os.path.join(DOSSIER_RESULTATS, "04b_sous_jeux_arbres.csv")
resultats_arbres = pd.DataFrame()
if os.path.isfile(chemin_arbres):
    resultats_arbres = pd.read_csv(chemin_arbres)
else:
    print("docs/resultats/04b_sous_jeux_arbres.csv absent : tableau sans les arbres.")

NOMS_MODELES_SOUS_JEUX = {"xgboost_phase3": "XGBoost Phase 3", "xgboost_retenu": "XGBoost retenu", "lstm": "LSTM", "cnn1d": "CNN 1D"}


def ligne_sous_jeu(r, protocole, rmse_validation):
    '''
    Construit une ligne du tableau par sous-jeu a partir d'une ligne de resultats (arbres ou reseaux).

    Parameters
    ----------
    r : pandas.Series
        La ligne de resultats (colonnes rmse_test, nasa_test, ...).
    protocole : str
        Le protocole de validation (texte court).
    rmse_validation : float
        Le RMSE de validation selon ce protocole.

    Returns
    -------
    dict
        La ligne du tableau.
    '''
    ligne = {}
    ligne["sous_jeu"] = r["sous_jeu"]
    ligne["modele"] = NOMS_MODELES_SOUS_JEUX[r["modele"]]
    ligne["rmse_test"] = r["rmse_test"]
    ligne["rmse_test_rul_plafonnee"] = r["rmse_test_rul_plafonnee"]
    ligne["nasa_test_par_moteur"] = r["nasa_test"] / r["nb_moteurs_test"]
    ligne["surestimes_test_20"] = int(r["surestimes_test"])
    ligne["alertes_manquees_test"] = int(r["alertes_manquees_test"])
    ligne["fausses_alertes_test"] = int(r["fausses_alertes_test"])
    ligne["nb_moteurs_test"] = int(r["nb_moteurs_test"])
    ligne["protocole_validation"] = protocole
    ligne["rmse_validation"] = rmse_validation
    return ligne

Le tableau : une ligne par (sous-jeu, modèle). Rappel : la colonne `rmse_validation` n'est pas comparable entre
arbres (validation croisée 5 plis) et réseaux sur FD002–FD004 (séparation 80/20, moteurs d'arrêt précoce) ; seules
les colonnes de test le sont.

In [ ]:
lignes = []
for _, r in resultats_arbres.iterrows():  # arbres : validation croisee 5 plis
    lignes.append(ligne_sous_jeu(r, "CV 5 plis", r["rmse_cv_moyen"]))
for _, r in resultats_dl.iterrows():  # reseaux : CV 5 plis (FD001) ou separation 80/20 (FD002-FD004)
    protocole_court = "séparation 80/20"
    if str(r["protocole_validation"]).startswith("validation croisee"):
        protocole_court = "CV 5 plis"
    lignes.append(ligne_sous_jeu(r, protocole_court, r["rmse_validation"]))
tableau_sous_jeux = pd.DataFrame(lignes)
ordre_modeles = ["XGBoost Phase 3", "XGBoost retenu", "LSTM", "CNN 1D"]
tableau_sous_jeux["ordre"] = 0
for position in range(len(ordre_modeles)):
    tableau_sous_jeux.loc[tableau_sous_jeux["modele"] == ordre_modeles[position], "ordre"] = position
tableau_sous_jeux = tableau_sous_jeux.sort_values(["sous_jeu", "ordre"]).drop(columns="ordre").set_index(["sous_jeu", "modele"])
tableau_sous_jeux.round(2)

### G.2 Tests appariés sur le test, sous-jeu par sous-jeu

Erreur absolue au dernier cycle de chaque moteur test, chaque réseau contre XGBoost Phase 3 (les fichiers
`03b_predictions_test_dernier_cycle.csv` et `04b_predictions_test_dernier_cycle.csv` ont une ligne par moteur
test). 8 comparaisons (2 réseaux × 4 sous-jeux) : seuil de Bonferroni 0,05 / 8 ≈ 0,006.

In [ ]:
derniers_reseaux = pd.read_csv(os.path.join(DOSSIER_RESULTATS, "03b_predictions_test_dernier_cycle.csv"))
chemin_derniers_arbres = os.path.join(DOSSIER_RESULTATS, "04b_predictions_test_dernier_cycle.csv")
SEUIL_BONFERRONI_SOUS_JEUX = 0.05 / 8
lignes = []
if os.path.isfile(chemin_derniers_arbres):
    derniers_arbres = pd.read_csv(chemin_derniers_arbres)
    for sous_jeu in ["FD001", "FD002", "FD003", "FD004"]:
        reference = derniers_arbres[(derniers_arbres["sous_jeu"] == sous_jeu) & (derniers_arbres["modele"] == "xgboost_phase3")]
        reference = reference.sort_values("unit_number")
        for nom_reseau in ["lstm", "cnn1d"]:
            reseau = derniers_reseaux[(derniers_reseaux["sous_jeu"] == sous_jeu) & (derniers_reseaux["modele"] == nom_reseau)]
            reseau = reseau.sort_values("unit_number")
            if len(reseau) == 0 or len(reference) == 0:
                continue
            assert np.array_equal(reseau["unit_number"].to_numpy(), reference["unit_number"].to_numpy())
            erreurs_reseau = np.abs(reseau["rul_predite"].to_numpy() - reseau["rul_vraie"].to_numpy())
            erreurs_reference = np.abs(reference["rul_predite"].to_numpy() - reference["rul_vraie"].to_numpy())
            resultat = comparer_apparie(erreurs_reseau, erreurs_reference)
            ligne = {}
            ligne["sous_jeu"] = sous_jeu
            ligne["comparaison"] = f"{NOMS_AFFICHES[nom_reseau]} vs XGBoost Phase 3"
            ligne["n_moteurs_test"] = resultat["n"]
            ligne["erreur_absolue_moyenne_reseau"] = float(np.mean(erreurs_reseau))
            ligne["erreur_absolue_moyenne_xgboost"] = float(np.mean(erreurs_reference))
            ligne["difference_moyenne"] = resultat["difference_moyenne"]
            ligne["moteurs_ou_le_reseau_est_meilleur"] = int(np.sum(erreurs_reseau < erreurs_reference))
            ligne["p_valeur_t"] = resultat["p_valeur_t"]
            ligne["p_valeur_wilcoxon"] = resultat["p_valeur_wilcoxon"]
            ligne["significatif_bonferroni"] = bool(resultat["p_valeur_t"] < SEUIL_BONFERRONI_SOUS_JEUX and resultat["p_valeur_wilcoxon"] < SEUIL_BONFERRONI_SOUS_JEUX)
            lignes.append(ligne)
pd.DataFrame(lignes).set_index(["sous_jeu", "comparaison"]).round(4)

### G.3 Validation croisée complémentaire : réseaux et arbres sur les mêmes plis

Même protocole qu'en FD001 (section D) : 5 plis = les plis des arbres du sous-jeu, un réseau neuf par pli, arrêt
précoce sur 20 % des moteurs d'**entraînement** du pli. Calculée à part par le script
(`--seulement-cv-sous-jeux`), elle n'a servi à aucun choix. Les colonnes sont celles de la Phase 4 ; le score NASA
CV est la moyenne, sur les plis, de la somme des pénalités de toutes les lignes de validation.

D'abord une petite fonction qui met une ligne de résultats (réseau ou arbre) au même format :

In [ ]:
def ligne_cv_sous_jeu(sous_jeu, nom_modele, r):
    '''
    Met une ligne de validation croisee (reseau ou arbre) au format du tableau par sous-jeu.

    Parameters
    ----------
    sous_jeu : str
        Le sous-jeu.
    nom_modele : str
        Le nom affiche du modele.
    r : pandas.Series
        La ligne de resultats (colonnes rmse_cv_moyen, nasa_cv_moyen, surestimes_oof, ...).

    Returns
    -------
    dict
        La ligne du tableau.
    '''
    ligne = {}
    ligne["sous_jeu"] = sous_jeu
    ligne["modele"] = nom_modele
    ligne["rmse_cv_moyen"] = r["rmse_cv_moyen"]
    ligne["rmse_cv_ecart_type"] = r["rmse_cv_ecart_type"]
    ligne["nasa_cv_moyen"] = r["nasa_cv_moyen"]
    ligne["surestimes_oof_20"] = r["surestimes_oof"]
    ligne["alertes_manquees_oof"] = r["alertes_manquees_oof"]
    ligne["fausses_alertes_oof"] = r["fausses_alertes_oof"]
    ligne["biais_oof"] = r["biais_oof"]
    return ligne

Le tableau : réseaux (`03b_cv_<sous-jeu>.csv`) et arbres (`04b_sous_jeux_arbres.csv`), sous-jeu par sous-jeu.

In [ ]:
lignes = []
for sous_jeu in ["FD001", "FD002", "FD003", "FD004"]:
    chemin_cv_sous_jeu = os.path.join(DOSSIER_RESULTATS, f"03b_cv_{sous_jeu.lower()}.csv")
    if os.path.isfile(chemin_cv_sous_jeu):  # reseaux (validation croisee faite par le script)
        cv_sous_jeu = pd.read_csv(chemin_cv_sous_jeu)
        for nom_reseau in ["lstm", "cnn1d"]:
            lignes_reseau = cv_sous_jeu[cv_sous_jeu["modele"] == nom_reseau]
            if len(lignes_reseau) > 0:
                lignes.append(ligne_cv_sous_jeu(sous_jeu, NOMS_AFFICHES[nom_reseau], lignes_reseau.iloc[0]))
    for _, r in resultats_arbres[resultats_arbres["sous_jeu"] == sous_jeu].iterrows():  # arbres
        lignes.append(ligne_cv_sous_jeu(sous_jeu, NOMS_MODELES_SOUS_JEUX[r["modele"]], r))
tableau_cv_sous_jeux = pd.DataFrame(lignes)
tableau_cv_sous_jeux["ordre"] = 0
for position in range(len(ordre_modeles)):
    tableau_cv_sous_jeux.loc[tableau_cv_sous_jeux["modele"] == ordre_modeles[position], "ordre"] = position
tableau_cv_sous_jeux = tableau_cv_sous_jeux.sort_values(["sous_jeu", "ordre"]).drop(columns="ordre").set_index(["sous_jeu", "modele"])
tableau_cv_sous_jeux.round(2)

### G.4 Tests appariés en validation croisée, sous-jeu par sous-jeu

RMSE hors-pli de **chaque moteur d'entraînement** (100, 260, 100 et 249 moteurs), chaque réseau contre XGBoost
Phase 3 (fichier `04b_metriques_par_moteur_cv.csv` des arbres, mêmes plis). Beaucoup plus de moteurs que sur le
test : ces tests sont plus sensibles. 8 comparaisons : seuil de Bonferroni 0,05 / 8 ≈ 0,006. Pour FD001, le RMSE
par moteur des réseaux est recalculé ici à partir des prédictions hors-pli (section D).

In [ ]:
chemin_par_moteur_reseaux = os.path.join(DOSSIER_RESULTATS, "03b_metriques_par_moteur_cv.csv")
chemin_par_moteur_arbres = os.path.join(DOSSIER_RESULTATS, "04b_metriques_par_moteur_cv.csv")
par_moteur_reseaux = pd.DataFrame()
if os.path.isfile(chemin_par_moteur_reseaux):
    par_moteur_reseaux = pd.read_csv(chemin_par_moteur_reseaux)
for nom_reseau in ["lstm", "cnn1d"]:  # FD001 : a partir des predictions hors-pli de la section D
    par_moteur_fd001 = metriques_par_moteur(y, oof[f"oof_{nom_reseau}"], groups)
    par_moteur_fd001.insert(0, "modele", nom_reseau)
    par_moteur_fd001.insert(0, "sous_jeu", "FD001")
    par_moteur_reseaux = pd.concat([par_moteur_reseaux, par_moteur_fd001], ignore_index=True)

lignes = []
if os.path.isfile(chemin_par_moteur_arbres):
    par_moteur_arbres = pd.read_csv(chemin_par_moteur_arbres)
    for sous_jeu in ["FD001", "FD002", "FD003", "FD004"]:
        reference = par_moteur_arbres[(par_moteur_arbres["sous_jeu"] == sous_jeu) & (par_moteur_arbres["modele"] == "xgboost_phase3")]
        reference = reference.sort_values("unit_number")
        for nom_reseau in ["lstm", "cnn1d"]:
            reseau = par_moteur_reseaux[(par_moteur_reseaux["sous_jeu"] == sous_jeu) & (par_moteur_reseaux["modele"] == nom_reseau)]
            reseau = reseau.sort_values("unit_number")
            if len(reseau) == 0 or len(reference) == 0:
                continue
            assert np.array_equal(reseau["unit_number"].to_numpy(), reference["unit_number"].to_numpy())
            rmse_reseau = reseau["rmse"].to_numpy(dtype=float)
            rmse_reference = reference["rmse"].to_numpy(dtype=float)
            resultat = comparer_apparie(rmse_reseau, rmse_reference)
            ligne = {}
            ligne["sous_jeu"] = sous_jeu
            ligne["comparaison"] = f"{NOMS_AFFICHES[nom_reseau]} vs XGBoost Phase 3"
            ligne["n_moteurs"] = resultat["n"]
            ligne["rmse_moyen_par_moteur_reseau"] = float(np.mean(rmse_reseau))
            ligne["rmse_moyen_par_moteur_xgboost"] = float(np.mean(rmse_reference))
            ligne["difference_moyenne"] = resultat["difference_moyenne"]
            ligne["moteurs_ou_le_reseau_est_meilleur"] = int(np.sum(rmse_reseau < rmse_reference))
            ligne["p_valeur_t"] = resultat["p_valeur_t"]
            ligne["p_valeur_wilcoxon"] = resultat["p_valeur_wilcoxon"]
            ligne["significatif_bonferroni"] = bool(resultat["p_valeur_t"] < SEUIL_BONFERRONI_SOUS_JEUX and resultat["p_valeur_wilcoxon"] < SEUIL_BONFERRONI_SOUS_JEUX)
            lignes.append(ligne)
pd.DataFrame(lignes).set_index(["sous_jeu", "comparaison"]).round(4)

### G.5 Figure : résultats par sous-jeu

Quatre panneaux : RMSE de validation croisée (mêmes plis, moyenne des 5 plis), RMSE test contre la RUL brute
(protocole officiel), RMSE test contre la RUL plafonnée (ce que les modèles pouvaient apprendre), score NASA par
moteur test.

In [ ]:
sous_jeux_figure = ["FD001", "FD002", "FD003", "FD004"]
modeles_figure = ["XGBoost Phase 3", "XGBoost retenu", "LSTM", "CNN 1D"]
couleurs_modeles_figure = [COULEUR_1, COULEUR_4, COULEUR_2, COULEUR_3]
largeur_barre = 0.2
tableau_figure = tableau_sous_jeux.join(tableau_cv_sous_jeux[["rmse_cv_moyen"]], how="left")
fig, axes = plt.subplots(1, 4, figsize=(22, 5.2))
panneaux = [("rmse_cv_moyen", "RMSE validation croisée 5 plis"), ("rmse_test", "RMSE test (RUL brute)"),
            ("rmse_test_rul_plafonnee", "RMSE test (RUL plafonnée à 125)"), ("nasa_test_par_moteur", "score NASA test par moteur")]
for ax, (colonne, titre) in zip(axes, panneaux):
    for position_modele in range(len(modeles_figure)):
        nom_modele = modeles_figure[position_modele]
        for position_sous_jeu in range(len(sous_jeux_figure)):
            cle = (sous_jeux_figure[position_sous_jeu], nom_modele)
            if cle not in tableau_figure.index:
                continue
            valeur = tableau_figure.loc[cle, colonne]
            if pd.isna(valeur):
                continue
            x = position_sous_jeu + (position_modele - 1.5) * largeur_barre
            etiquette = None
            if position_sous_jeu == 0:
                etiquette = nom_modele
            ax.bar(x, valeur, width=largeur_barre, color=couleurs_modeles_figure[position_modele], label=etiquette)
            ax.text(x, valeur, f"{valeur:.1f}", ha="center", va="bottom", fontsize=7, rotation=90)
    ax.set_xticks(np.arange(len(sous_jeux_figure)))
    ax.set_xticklabels(sous_jeux_figure)
    ax.set_title(titre, fontsize=10)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.12)  # un peu de place au-dessus des barres pour les valeurs
# une seule legende pour les quatre panneaux, sous les graphiques (elle ne cache aucune barre)
poignees, etiquettes = axes[0].get_legend_handles_labels()
fig.legend(poignees, etiquettes, loc="lower center", ncol=4, fontsize=9, frameon=False)
fig.suptitle("Quatre sous-jeux : validation croisée (mêmes plis) et test officiel, arbres et réseaux (plus bas = mieux)")
plt.tight_layout(rect=(0, 0.06, 1, 1))  # on laisse une bande en bas pour la legende
plt.savefig(os.path.join(DOSSIER_FIGURES, "03b_resultats_par_sous_jeu.png"), dpi=150)
plt.show()

### G.6 Conclusion de la section G

- **Test officiel : le LSTM a le plus petit RMSE des quatre modèles sur les quatre sous-jeux.** FD002 : 25,45
  contre 27,23 (XGBoost Phase 3) ; FD003 : 16,80 contre 18,09 ; FD004 : 26,14 contre 28,24 (FD001 : 13,86 contre
  19,70). Contre la RUL plafonnée : 12,73 contre 15,46 (FD002), 15,29 contre 16,66 (FD003), 14,22 contre 17,28
  (FD004). Test apparié sur l'erreur absolue par moteur test, contre XGBoost Phase 3 : significatif (Bonferroni)
  sur FD002 (−2,16 cycles, p = 0,0001 / 0,0037) et FD004 (−2,74, p < 0,0001) ; **non significatif sur FD003**
  (−1,16, p = 0,30 / 0,83).
- **Score NASA : pas meilleur partout.** Par moteur test, le LSTM fait mieux que XGBoost Phase 3 sur FD002 (31,3
  contre 35,7) et FD003 (5,7 contre 6,9), mais **moins bien sur FD004** (27,3 contre 22,3 ; XGBoost retenu : 33,4).
  Alertes manquées : 1 / 3 / 3 (FD002 / FD003 / FD004) contre 2 / 3 / 7 pour XGBoost Phase 3.
- **Validation croisée, mêmes plis que les arbres** : le LSTM a le plus petit RMSE CV partout — FD002 14,02 ± 0,50
  contre 16,41 ± 0,77 ; FD003 13,59 ± 2,11 contre 14,82 ± 1,79 ; FD004 14,64 ± 2,17 contre 15,93 ± 1,73. RMSE par
  moteur contre XGBoost Phase 3 : FD002 −1,87 (p < 0,0001, 169 moteurs sur 260 en faveur du LSTM) ; FD003 −1,16
  (p = 0,008 / 0,006, juste au-dessus du seuil de Bonferroni de 0,006 : non significatif à ce seuil) ;
  FD004 −1,15 (p = 0,0004 / 0,0001, 148 moteurs sur 249).
- **Le CNN 1D reste en dessous de XGBoost** : test moins bon sur FD002 (+3,46 cycles par moteur) et FD003 (+3,13),
  significatif ; à égalité sur FD004 (−0,24, p = 0,68), mais avec 14 alertes manquées et 45 moteurs surestimés de
  plus de 20 cycles. En validation croisée, il est moins bon que XGBoost Phase 3 sur FD002 (+0,82 par moteur,
  p = 0,004 / 0,003) et sur FD004 (+1,61, p < 0,0001) ; sur FD003,
  +1,08 (p = 0,009 / 0,008, au-dessus du seuil de Bonferroni).
- **Limites** : les modèles publiés FD002–FD004 ont leur nombre d'époques fixé sur une séparation unique (80/20) ;
  architectures et réglages décidés sur FD001, non réoptimisés ; sur FD002 et FD004, le RMSE contre la RUL brute est
  dominé par les moteurs test de RUL vraie très supérieure à 125, qu'aucun modèle ne peut prédire.

## H. Synthèse

### H.1 Ce qui est établi (mêmes plis, tests appariés, test évalué une seule fois)

- **Sur FD001, le LSTM bat XGBoost**, en validation croisée (RMSE 13,70 contre 16,87 ; −2,55 cycles par moteur,
  p < 0,0001) comme sur le test officiel (RMSE 13,86 contre 19,70 et 19,73 ; score NASA 293 contre 853 et 675 ;
  0 alerte manquée contre 7 ; erreur absolue par moteur −4,0 cycles, p < 0,006), et sur 3 graines (RMSE test
  14,22 ± 0,30). Ses réglages ont été fixés à l'avance, alors que XGBoost retenu a été choisi parmi 160 essais
  Optuna : la comparaison avantage XGBoost, pas le LSTM.
- **Sur FD002 à FD004**, le LSTM a aussi le plus petit RMSE, en validation croisée comme sur le test ; l'écart est
  significatif sur FD002 et FD004, pas sur FD003 ; son score NASA test par moteur est moins bon que celui de XGBoost
  Phase 3 sur FD004.
- **Pénaliser les surestimations dans la perte** (α = 2,26) rend le LSTM plus prudent en validation croisée
  (−38 % de surestimations > 20 cycles, score NASA −21 %), sans perte de RMSE détectable, au prix de 83 % de
  fausses alertes en plus.
- **Le CNN 1D ne fait pas mieux que XGBoost** : à égalité sur FD001 en validation croisée, moins bon ailleurs, et
  instable d'une graine à l'autre sur le test (RMSE de 17,3 à 21,3).
- **Reproductible** : prédictions identiques au bit près à graine égale (CPU), poids rechargeables
  (`models/FD00X__lstm.pt`, `__cnn1d.pt`), chiffres XGBoost des Phases 3 et 4 retrouvés à l'identique ; toute la
  chaîne est testée (`tests/test_deep_learning.py` : fenêtres sans fuite, formes, apprentissage, reproductibilité,
  sauvegarde).

### H.2 Limites et coûts

- **Coût de calcul** : plus de 100 fois plus de temps d'entraînement que XGBoost (FD001, 5 plis : 385 s contre 2 à
  4 s) et plus de 100 fois plus de temps de prédiction (7 ms par moteur test contre moins de 0,1 ms), sur 1 cœur.
- **Une seule architecture par famille, sans recherche d'hyperparamètres** ; la décision sur `time_cycles` repose
  sur une séparation et 20 moteurs.
- **Le scaler min-max est appris sur tout le train avant la validation croisée** (comme pour les arbres, Phase 3).
- **Le modèle final FD001 voit environ 1,5 fois plus de lots par époque qu'en validation croisée** (100 moteurs au
  lieu de 64), pour un nombre d'époques fixé en validation croisée.
- **Explicabilité** : SHAP sur XGBoost est direct (Phase 5) ; sur un réseau séquentiel, il faut d'autres
  méthodes, pas encore étudiées dans le projet.

### H.3 Ce qui reste à décider par l'équipe

- Garder XGBoost retenu comme modèle de la démo, ou passer au LSTM (plus précis et moins d'alertes manquées sur
  FD001, mais plus coûteux et moins simple à expliquer) ? Si le modèle retenu change, une ADR doit l'acter
  (règle 9 : remplacement justifié par un progrès mesuré).
- Pour le LSTM : α = 1 ou α = 2,26 ? À trancher par une règle écrite avant de regarder le test, comme en Phase 4.
- `docs/03_modelisation.md` / `docs/04_evaluation.md` ne sont pas mis à jour ici (règle 4 : après validation).

### H.4 Fichiers produits

- `docs/resultats/03b_*` : choix des entrées, mesure CPU / MPS, validations croisées (`03b_cv_fd00X.csv`),
  prédictions hors-pli FD001, historiques d'apprentissage, résultats de test, tableau agrégé `03b_deep_learning.csv`,
  prédictions au dernier cycle des quatre sous-jeux ; `docs/figures/03b_*` (7 figures de ce notebook).
- `sorties/predictions/FD00X__lstm.*` et `FD00X__cnn1d.*` (format de `src/contrat_predictions.py`, pour le tableau
  de bord) et `models/FD00X__lstm.pt`, `FD00X__cnn1d.pt` : produits par le script, non versionnés.

In [ ]:
duree_notebook = time.perf_counter() - debut_notebook
print(f"Notebook execute en {duree_notebook / 60:.1f} minutes.")